# BTXRD Dataset — Portion 3 + 4 + 5 Final T4-Efficient SOTA Notebook v5

**Dataset name:** `BTXRD dataset`  
**Project:** Primary bone tumor radiograph diagnosis  
**Final model:** `BTX-TrustNet-LiteSOTA-v5`

This notebook is rebuilt to solve the Kaggle runtime failure from the previous Portion 3/4/5 run. The old run failed because it could not find a rigid path like `BTXRD/images`. This version does **not** depend on one hard-coded raw-root path. It first uses the corrected Portion-2 split/master if available, repairs stale file paths by basename, and then scans all attached Kaggle inputs for valid radiograph files.

---

## Planwise structure

| Portion | Goal | Main output |
|---:|---|---|
| **3** | Build the final T4-efficient SOTA architecture | `BTX-TrustNet-LiteSOTA-v5` |
| **4** | Accuracy-focused training and optimization | calibrated validation-weighted ensemble |
| **5** | Journal-level evaluation package | 30+ metrics, 35+ figures, final report, ZIP export |

---

## Kaggle inputs expected

Keep your existing inputs:

```python
PORTION1_INPUT = Path('/kaggle/input/notebooks/koushikrudra/btxrd-p1')
PORTION2_INPUT = Path('/kaggle/input/datasets/koushikrudra/btxrd-p2')
```

Also attach the original Kaggle dataset named **BTXRD dataset**. The notebook searches for it automatically under `/kaggle/input`.

---

## T4-efficient design decision

The previous architecture was too heavy and too path-sensitive. This version uses a protected accuracy design:

```text
EfficientNet-B3 global expert
        +
Lesion-aware TrustNet expert
        +
validation-only ensemble weighting
        +
temperature calibration
        +
logit-offset operating point tuning
```

The model is designed to be strong on a Tesla T4 without depending on DINOv2/SAM downloads.


In [1]:
# ============================================================
# Cell 1 — Global configuration and environment setup
# ============================================================
import os, re, json, math, time, random, shutil, warnings, hashlib, gc, zipfile
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps, ImageEnhance

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, log_loss, brier_score_loss, cohen_kappa_score, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import functional as TF

try:
    from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
except Exception:
    efficientnet_b3, EfficientNet_B3_Weights = None, None

try:
    from IPython.display import display
except Exception:
    display = print

# ----------------------------
# User paths
# ----------------------------
PORTION1_INPUT = Path('/kaggle/input/notebooks/koushikrudra/btxrd-p1')
PORTION2_INPUT = Path('/kaggle/input/datasets/koushikrudra/btxrd-p2')
DATASET_NAME_HINT = 'BTXRD dataset'

# ----------------------------
# Run profile
# ----------------------------
# smoke    : validates paths and code quickly.
# t4_sota  : default; designed to finish on Kaggle Tesla T4.
# full_sota: slower; use only if you have long runtime and want 3-seed 384px ensemble.
RUN_PROFILE = 't4_sota'   # 'smoke', 't4_sota', or 'full_sota'
RUN_MODE = 'publication' if RUN_PROFILE != 'smoke' else 'smoke'

# No internet assumption: do not let torchvision spend runtime trying to download weights.
# The notebook instead searches Portion-2 for your trained EfficientNet-B3 checkpoint.
ALLOW_TORCHVISION_WEIGHT_DOWNLOAD = False

if RUN_PROFILE == 'smoke':
    CFG = dict(
        image_size=224, batch_size=8, epochs_global=1, epochs_trust=1, freeze_epochs=0,
        seeds=[42], num_workers=0, bootstrap_n=50, max_train_rows=None, amp=True, early_stop=2,
        train_global_if_no_p2=True, train_trustnet=True, use_weighted_sampler=True, tta=False,
    )
elif RUN_PROFILE == 'full_sota':
    CFG = dict(
        image_size=384, batch_size=10, epochs_global=8, epochs_trust=14, freeze_epochs=2,
        seeds=[42, 909, 2025], num_workers=0, bootstrap_n=700, max_train_rows=None, amp=True, early_stop=5,
        train_global_if_no_p2=True, train_trustnet=True, use_weighted_sampler=True, tta=False,
    )
else:  # t4_sota
    CFG = dict(
        image_size=320, batch_size=16, epochs_global=4, epochs_trust=10, freeze_epochs=1,
        seeds=[42, 909], num_workers=0, bootstrap_n=400, max_train_rows=None, amp=True, early_stop=3,
        train_global_if_no_p2=True, train_trustnet=True, use_weighted_sampler=True, tta=False,
    )

# ----------------------------
# Output folders
# ----------------------------
OUT_DIR = Path('/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5')
TABLE_DIR = OUT_DIR / 'tables'
FIG_DIR = OUT_DIR / 'figures'
MODEL_DIR = OUT_DIR / 'models'
REPORT_DIR = OUT_DIR / 'reports'
ARR_DIR = OUT_DIR / 'arrays'
for d in [OUT_DIR, TABLE_DIR, FIG_DIR, MODEL_DIR, REPORT_DIR, ARR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Reproducibility and device
# ----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('RUN_PROFILE:', RUN_PROFILE)
print('RUN_MODE:', RUN_MODE)
print('CFG:', json.dumps(CFG, indent=2))
print('Output:', OUT_DIR)

# ----------------------------
# Journal style
# ----------------------------
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 400,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

CLASS_NAMES = ['normal', 'benign', 'malignant']
CLASS_TO_ID = {c:i for i,c in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i:c for c,i in CLASS_TO_ID.items()}
TUMOR_CLASSES = ['benign', 'malignant']


Device: cuda
GPU: Tesla T4
RUN_PROFILE: t4_sota
RUN_MODE: publication
CFG: {
  "image_size": 320,
  "batch_size": 16,
  "epochs_global": 4,
  "epochs_trust": 10,
  "freeze_epochs": 1,
  "seeds": [
    42,
    909
  ],
  "num_workers": 0,
  "bootstrap_n": 400,
  "max_train_rows": null,
  "amp": true,
  "early_stop": 3,
  "train_global_if_no_p2": true,
  "train_trustnet": true,
  "use_weighted_sampler": true,
  "tta": false
}
Output: /kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5


# Portion 3/4/5 — Step 1: Robust input discovery and data repair

The previous notebook failed because it required `RAW_ROOT/images` to exist. This version uses a more reliable plan:

1. Extract Portion-1 ZIP if needed.
2. Search Portion-2 for corrected split files or all-class master CSV.
3. Repair image and mask paths using a basename index built from all valid Kaggle inputs.
4. If no corrected split is available, rebuild the all-class dataset from tumor tables plus normal radiographs found anywhere under `/kaggle/input`.
5. Stop only if the notebook cannot resolve enough actual X-ray images for a true 3-class experiment.


In [2]:
# ============================================================
# Cell 2 — Input discovery, ZIP extraction, and file indexing
# ============================================================
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
TABLE_EXTS = {'.csv', '.xlsx', '.xls'}

BAD_IMAGE_PATH_PATTERNS = [
    'figures', 'figure', 'report', 'reports', 'confusion', 'roc', 'precision_recall', 'pr_curve',
    'reliability', 'architecture', 'sample', 'overlay', 'xai', 'heatmap', 'mask_overlay',
    'metrics', 'plots', 'plot', 'loss', 'curve', 'gradcam', 'cam_', 'tsne', 'umap'
]
MASK_PATH_PATTERNS = ['mask', 'masks_png', 'segmentation', 'annotations']

def pstr(p):
    return str(p).replace('\\', '/')

def is_probably_raw_xray_path(path: Path) -> bool:
    s = pstr(path).lower()
    if path.suffix.lower() not in IMAGE_EXTS:
        return False
    # Exclude generated visuals and mask folders.
    if any(tok in s for tok in BAD_IMAGE_PATH_PATTERNS):
        return False
    # Raw X-rays usually sit under images/image/img/radiograph folder. Still allow direct dataset folders.
    return True

def is_probably_mask_path(path: Path) -> bool:
    s = pstr(path).lower()
    if path.suffix.lower() not in IMAGE_EXTS:
        return False
    return any(tok in s for tok in MASK_PATH_PATTERNS) and not any(tok in s for tok in ['confusion','roc','reliability','architecture'])

def stem_key(x) -> str:
    if pd.isna(x): return ''
    p = Path(str(x))
    return p.stem.lower().strip()

def base_key(x) -> str:
    if pd.isna(x): return ''
    p = Path(str(x))
    return p.name.lower().strip()

def robust_existing_path(x) -> Optional[Path]:
    if x is None or pd.isna(x):
        return None
    s = str(x)
    if not s or s.lower() == 'nan':
        return None
    p = Path(s)
    if p.exists():
        return p
    return None

def extract_zip_safely(zip_path: Path, dest: Path) -> Optional[Path]:
    try:
        marker = dest / '.unzipped_ok'
        if marker.exists():
            return dest
        dest.mkdir(parents=True, exist_ok=True)
        print('Extracting ZIP:', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(dest)
        marker.write_text('ok')
        return dest
    except Exception as e:
        print('ZIP extraction failed:', zip_path, e)
        return None

# Gather roots from P1/P2 and their ZIP outputs.
P1_ROOTS = []
P2_ROOTS = []
if PORTION1_INPUT.exists():
    P1_ROOTS.append(PORTION1_INPUT)
if PORTION2_INPUT.exists():
    P2_ROOTS.append(PORTION2_INPUT)

# Unzip known output zips from Portion 1/2 if present.
for base, root_list, tag in [(PORTION1_INPUT, P1_ROOTS, 'p1'), (PORTION2_INPUT, P2_ROOTS, 'p2')]:
    if base.exists():
        for zp in list(base.rglob('*.zip'))[:12]:
            dest = OUT_DIR / f'_{tag}_unzipped' / zp.stem
            out = extract_zip_safely(zp, dest)
            if out is not None:
                root_list.append(out)

# Add common Kaggle input roots and any dataset name hints.
KAGGLE_INPUT = Path('/kaggle/input')
ALL_SEARCH_ROOTS = []
for p in [PORTION2_INPUT, PORTION1_INPUT, KAGGLE_INPUT] + P2_ROOTS + P1_ROOTS:
    if p and Path(p).exists():
        ALL_SEARCH_ROOTS.append(Path(p))
# unique, shallow-first
uniq = []
seen = set()
for p in ALL_SEARCH_ROOTS:
    s = str(p.resolve()) if p.exists() else str(p)
    if s not in seen:
        seen.add(s); uniq.append(p)
ALL_SEARCH_ROOTS = uniq

print('Portion-1 input:', PORTION1_INPUT, 'exists=', PORTION1_INPUT.exists())
print('Portion-2 input:', PORTION2_INPUT, 'exists=', PORTION2_INPUT.exists())
print('Search roots:')
for r in ALL_SEARCH_ROOTS:
    print(' -', r)

# Build basename/stem index for images and masks from /kaggle/input and extracted zips.
def build_file_indices(roots: List[Path], max_files: int = 120000):
    image_by_base = defaultdict(list)
    image_by_stem = defaultdict(list)
    mask_by_base = defaultdict(list)
    mask_by_stem = defaultdict(list)
    table_paths = []
    root_counts = Counter()
    n = 0
    for root in roots:
        if not root.exists():
            continue
        try:
            iterator = root.rglob('*')
            for p in iterator:
                if n > max_files:
                    break
                if not p.is_file():
                    continue
                n += 1
                ext = p.suffix.lower()
                if ext in TABLE_EXTS:
                    table_paths.append(p)
                elif ext in IMAGE_EXTS:
                    # root count for diagnostics
                    parts = p.parts
                    short_root = '/'.join(parts[:5]) if len(parts) >= 5 else str(p.parent)
                    root_counts[short_root] += 1
                    if is_probably_mask_path(p):
                        mask_by_base[p.name.lower()].append(p)
                        mask_by_stem[p.stem.lower()].append(p)
                    elif is_probably_raw_xray_path(p):
                        image_by_base[p.name.lower()].append(p)
                        image_by_stem[p.stem.lower()].append(p)
        except Exception as e:
            print('Index warning:', root, e)
    return image_by_base, image_by_stem, mask_by_base, mask_by_stem, table_paths, root_counts

image_by_base, image_by_stem, mask_by_base, mask_by_stem, table_paths, root_counts = build_file_indices(ALL_SEARCH_ROOTS)
print('Indexed raw-like image basenames:', len(image_by_base), '| mask basenames:', len(mask_by_base), '| tables:', len(table_paths))
print('Top image-containing roots:')
for k, v in root_counts.most_common(15):
    print(f'{v:6d}  {k}')

pd.DataFrame(root_counts.most_common(), columns=['root_prefix','image_file_count']).to_csv(TABLE_DIR/'input_image_root_audit.csv', index=False)
pd.DataFrame({'table_path':[str(p) for p in table_paths]}).to_csv(TABLE_DIR/'input_table_audit.csv', index=False)


Extracting ZIP: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1_outputs.zip
Portion-1 input: /kaggle/input/notebooks/koushikrudra/btxrd-p1 exists= True
Portion-2 input: /kaggle/input/datasets/koushikrudra/btxrd-p2 exists= True
Search roots:
 - /kaggle/input/datasets/koushikrudra/btxrd-p2
 - /kaggle/input/notebooks/koushikrudra/btxrd-p1
 - /kaggle/input
 - /kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/_p1_unzipped/BTXRD_Portion1_outputs
Indexed raw-like image basenames: 3757 | mask basenames: 1869 | tables: 161
Top image-containing roots:
  3778  //kaggle/input/notebooks/koushikrudra
  3746  //kaggle/input/datasets/koushikrudra
  3736  //kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/_p1_unzipped


In [3]:
# ============================================================
# Cell 3 — Robust CSV loaders, label normalization, path repair
# ============================================================
def read_table(path: Path) -> Optional[pd.DataFrame]:
    try:
        if path.suffix.lower() == '.csv':
            return pd.read_csv(path)
        if path.suffix.lower() in {'.xlsx', '.xls'}:
            return pd.read_excel(path)
    except Exception as e:
        print('Could not read table:', path, e)
    return None

LABEL_SYNONYMS = {
    'normal': 'normal', 'healthy': 'normal', 'negative': 'normal', '0': 'normal',
    'benign': 'benign', 'benign tumor': 'benign', 'benign_tumor': 'benign', '1': 'benign',
    'malignant': 'malignant', 'malignant tumor': 'malignant', 'malignant_tumor': 'malignant', '2': 'malignant',
}

def normalize_label_value(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return LABEL_SYNONYMS.get(s, s if s in CLASS_TO_ID else np.nan)

def infer_label_column(df: pd.DataFrame) -> Optional[str]:
    preferred = ['label_3class', 'class', 'label', 'diagnosis', 'target', 'category', 'type', 'disease_type']
    low = {c.lower(): c for c in df.columns}
    for key in preferred:
        if key in low:
            return low[key]
    # columns that contain mostly class strings
    for c in df.columns:
        vals = df[c].dropna().astype(str).str.lower().head(500).tolist()
        if not vals: continue
        hit = sum(any(tok in v for tok in ['normal','benign','malignant']) for v in vals)
        if hit >= max(5, 0.25*len(vals)):
            return c
    return None

def add_normalized_label(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'label_3class' in df.columns:
        lab = df['label_3class'].apply(normalize_label_value)
    else:
        c = infer_label_column(df)
        lab = df[c].apply(normalize_label_value) if c else pd.Series([np.nan]*len(df), index=df.index)
    # Indicator column fallback
    col_low = {str(c).lower(): c for c in df.columns}
    for idx in df.index[lab.isna()]:
        row = df.loc[idx]
        # explicit boolean columns
        for cls in CLASS_NAMES:
            candidate_cols = [c for c in df.columns if cls in str(c).lower()]
            for c in candidate_cols:
                val = row[c]
                try:
                    if str(val).strip().lower() in ['1','true','yes',cls]:
                        lab.loc[idx] = cls
                        break
                except Exception:
                    pass
            if pd.notna(lab.loc[idx]):
                break
    df['label_3class'] = lab
    return df

def find_path_col(df: pd.DataFrame, kind='image') -> Optional[str]:
    keys = ['image_path','img_path','path','filepath','file_path','filename','file','image','img','image_id','id'] if kind=='image' else ['mask_path','seg_path','mask','annotation_mask','mask_file','segmentation_path']
    low = {str(c).lower(): c for c in df.columns}
    for k in keys:
        if k in low:
            return low[k]
    for c in df.columns:
        name = str(c).lower()
        if kind == 'image' and any(tok in name for tok in ['image','img','file','path']):
            return c
        if kind == 'mask' and any(tok in name for tok in ['mask','seg']):
            return c
    return None

def best_candidate(cands: List[Path], prefer_mask=False) -> Optional[Path]:
    if not cands:
        return None
    def score(p: Path):
        s = pstr(p).lower()
        sc = 0
        if prefer_mask:
            if 'masks_png' in s or 'mask' in s: sc += 10
            if 'annotation' in s: sc += 4
        else:
            if '/images/' in s or s.endswith('/images'): sc += 10
            if 'btxrd' in s: sc += 4
            if any(tok in s for tok in BAD_IMAGE_PATH_PATTERNS): sc -= 100
            if 'mask' in s: sc -= 50
        # Prefer files from /kaggle/input over generated extraction unless all else equal.
        if '/kaggle/input/' in s: sc += 2
        return sc
    return sorted(cands, key=score, reverse=True)[0]

def resolve_file_value(v, prefer_mask=False) -> Optional[str]:
    if v is None or pd.isna(v): return None
    p = Path(str(v))
    if p.exists() and p.is_file():
        return str(p)
    keys = [base_key(v), stem_key(v)]
    if prefer_mask:
        for k in keys:
            cands = mask_by_base.get(k, []) if '.' in k else mask_by_stem.get(k, [])
            cand = best_candidate(cands, prefer_mask=True)
            if cand: return str(cand)
    else:
        for k in keys:
            cands = image_by_base.get(k, []) if '.' in k else image_by_stem.get(k, [])
            cand = best_candidate(cands, prefer_mask=False)
            if cand: return str(cand)
    # Try common extension swaps by stem.
    st = stem_key(v)
    if st:
        cands = (mask_by_stem if prefer_mask else image_by_stem).get(st, [])
        cand = best_candidate(cands, prefer_mask=prefer_mask)
        if cand: return str(cand)
    return None

def repair_paths(df: pd.DataFrame, verbose=True) -> pd.DataFrame:
    df = df.copy()
    img_col = find_path_col(df, 'image')
    mask_col = find_path_col(df, 'mask')
    if img_col is None:
        # Try constructing from image_id/name-like columns.
        for c in df.columns:
            if str(c).lower() in ['image_id','id','name','filename']:
                img_col = c; break
    if img_col is None:
        df['image_path'] = None
    else:
        df['image_path'] = df[img_col].apply(lambda x: resolve_file_value(x, prefer_mask=False))
    if mask_col is not None:
        df['mask_path'] = df[mask_col].apply(lambda x: resolve_file_value(x, prefer_mask=True))
    elif 'mask_path' not in df.columns:
        # resolve by image stem for tumor rows
        df['mask_path'] = df[img_col].apply(lambda x: resolve_file_value(x, prefer_mask=True)) if img_col is not None else None
    else:
        df['mask_path'] = df['mask_path'].apply(lambda x: resolve_file_value(x, prefer_mask=True))
    # If mask unresolved but image stem exists, try by image stem.
    unresolved = df['mask_path'].isna()
    if 'image_path' in df.columns:
        df.loc[unresolved, 'mask_path'] = df.loc[unresolved, 'image_path'].apply(lambda x: resolve_file_value(Path(str(x)).stem if pd.notna(x) else x, prefer_mask=True))
    df['image_stem'] = df['image_path'].apply(lambda x: Path(str(x)).stem.lower() if pd.notna(x) else '')
    df['has_image'] = df['image_path'].apply(lambda x: isinstance(x, str) and Path(x).exists())
    df['has_mask'] = df['mask_path'].apply(lambda x: isinstance(x, str) and Path(x).exists())
    if verbose:
        print('Path repair:', len(df), 'rows | images:', int(df['has_image'].sum()), '| masks:', int(df['has_mask'].sum()))
    return df

def dataset_ok(df: pd.DataFrame, min_normal=1000, require_splits=False) -> bool:
    if df is None or len(df)==0 or 'label_3class' not in df.columns:
        return False
    d = add_normalized_label(df)
    if 'has_image' not in d.columns:
        d = repair_paths(d, verbose=False)
    counts = d[d['has_image']]['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    return counts['normal'] >= min_normal and counts['benign'] > 100 and counts['malignant'] > 20


In [4]:
# ============================================================
# Cell 4 — Build or load final 3-class dataset/splits
# ============================================================
def find_tables_by_names(names: List[str], roots: List[Path]) -> List[Path]:
    out = []
    wanted = [n.lower() for n in names]
    for p in table_paths:
        pl = p.name.lower()
        if any(w in pl for w in wanted):
            out.append(p)
    # Prefer P2, then P1, then others.
    def rank(p):
        s = pstr(p).lower()
        r = 0
        if 'btxrd-p2' in s or 'portion2' in s or 'p2' in s: r -= 10
        if 'btxrd-p1' in s or 'portion1' in s or 'p1' in s: r -= 5
        if 'working' in s: r -= 2
        return (r, len(s))
    return sorted(set(out), key=rank)

def try_load_p2_splits() -> Optional[Dict[str,pd.DataFrame]]:
    # Search exact split names first.
    split_candidates = {}
    for split in ['train','val','test']:
        patterns = [f'final_internal_{split}.csv', f'gold_multitask_{split}.csv', f'*{split}.csv']
        hits = []
        for p in table_paths:
            name = p.name.lower()
            if f'final_internal_{split}' in name or f'allclass_{split}' in name or f'p345_{split}' in name:
                hits.append(p)
        # Prefer P2 corrected split.
        hits = sorted(set(hits), key=lambda p: (0 if 'btxrd-p2' in pstr(p).lower() else 1, len(pstr(p))))
        if hits:
            split_candidates[split] = hits[0]
    if not all(k in split_candidates for k in ['train','val','test']):
        return None
    print('Found split candidates:')
    for k,p in split_candidates.items(): print(' ', k, p)
    splits = {}
    for split,p in split_candidates.items():
        df = read_table(p)
        if df is None: return None
        df = add_normalized_label(df)
        df = repair_paths(df, verbose=False)
        df = df[df['has_image'] & df['label_3class'].isin(CLASS_NAMES)].copy()
        df['split'] = split
        splits[split] = df
        print(f'{split} counts:', df['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict(), '| n=',len(df))
    full = pd.concat(splits.values(), ignore_index=True)
    counts = full['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    if counts['normal'] >= 1000 and counts['benign'] > 100 and counts['malignant'] > 20:
        # Every split must contain all classes.
        ok = True
        for split,df in splits.items():
            sc = df['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
            if (sc <= 0).any(): ok = False
        if ok:
            return splits
    print('P2 split candidates are incomplete after path repair; rebuilding/looking for master.')
    return None

def try_load_allclass_master() -> Optional[pd.DataFrame]:
    keywords = ['btxrd_final_gold_plus_normals', 'p345_master_allclass', 'allclass', 'gold_plus_normals', 'classification_all_valid']
    candidates = []
    for p in table_paths:
        name = p.name.lower()
        if any(k in name for k in keywords):
            candidates.append(p)
    candidates = sorted(set(candidates), key=lambda p: (0 if 'btxrd-p2' in pstr(p).lower() else 1, len(pstr(p))))
    for p in candidates:
        df = read_table(p)
        if df is None or len(df)==0: continue
        df = add_normalized_label(df)
        df = repair_paths(df, verbose=False)
        df = df[df['has_image'] & df['label_3class'].isin(CLASS_NAMES)].copy()
        counts = df['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
        print('Master candidate:', p, counts.to_dict())
        if counts['normal'] >= 1000 and counts['benign'] > 100 and counts['malignant'] > 20:
            return df
    return None

def find_tumor_gold_table() -> Optional[pd.DataFrame]:
    candidates = []
    for p in table_paths:
        name = p.name.lower()
        if 'gold_multitask' in name or 'master_gold' in name or 'btxrd_master_gold' in name:
            candidates.append(p)
    candidates = sorted(set(candidates), key=lambda p: (0 if 'btxrd-p1' in pstr(p).lower() else 1, len(pstr(p))))
    for p in candidates:
        df = read_table(p)
        if df is None or len(df)==0: continue
        df = add_normalized_label(df)
        df = repair_paths(df, verbose=False)
        df = df[df['has_image'] & df['label_3class'].isin(['benign','malignant'])].copy()
        counts = df['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
        print('Tumor candidate:', p, counts.to_dict())
        if counts['benign'] > 100 and counts['malignant'] > 20:
            return df
    return None

def recover_normals_from_tables_and_images(tumor_df: pd.DataFrame) -> pd.DataFrame:
    tumor_stems = set(tumor_df['image_stem'].dropna().astype(str).str.lower())
    normal_frames = []
    # Table-based normal recovery: rows not in tumor stems, unknown/normal labels, image exists.
    for p in table_paths:
        name = p.name.lower()
        if not any(tok in name for tok in ['all_audited','all_valid','classification','dataset','master']):
            continue
        df = read_table(p)
        if df is None or len(df)==0: continue
        df = add_normalized_label(df)
        df = repair_paths(df, verbose=False)
        df = df[df['has_image']].copy()
        df['image_stem'] = df['image_path'].apply(lambda x: Path(str(x)).stem.lower())
        # Safe normal rule: explicit normal OR missing label and not tumor stem, no mask.
        candidates = df[(df['label_3class'].eq('normal')) | (df['label_3class'].isna())].copy()
        candidates = candidates[~candidates['image_stem'].isin(tumor_stems)].copy()
        # avoid annotation/mask rows
        candidates = candidates[~candidates['has_mask']].copy()
        if len(candidates):
            candidates['label_3class'] = 'normal'
            candidates['normal_source'] = f'table:{p.name}'
            normal_frames.append(candidates)
            print('Recovered table normals from:', p, len(candidates))
    # Image-scan normal recovery: all raw-like images not in tumor stems and not masks/figures.
    raw_paths = []
    for cands in image_by_stem.values():
        for p in cands:
            if p.stem.lower() not in tumor_stems and is_probably_raw_xray_path(p):
                raw_paths.append(p)
    raw_paths = list(dict.fromkeys(raw_paths))
    if raw_paths:
        rows = []
        for p in raw_paths:
            rows.append({'image_path': str(p), 'mask_path': None, 'label_3class': 'normal', 'image_stem': p.stem.lower(), 'has_image': True, 'has_mask': False, 'normal_source': 'raw_scan'})
        raw_df = pd.DataFrame(rows)
        print('Recovered raw-scan normal candidates:', len(raw_df))
        normal_frames.append(raw_df)
    if not normal_frames:
        return pd.DataFrame()
    normal_df = pd.concat(normal_frames, ignore_index=True, sort=False)
    normal_df = normal_df.drop_duplicates(subset=['image_stem']).copy()
    # Optional sanity filter: open a sample of files; then validate all dimensions cheaply.
    def valid_image(path):
        try:
            with Image.open(path) as im:
                w,h = im.size
            return w >= 128 and h >= 128
        except Exception:
            return False
    normal_df['valid_image_open'] = normal_df['image_path'].apply(valid_image)
    normal_df = normal_df[normal_df['valid_image_open']].copy()
    return normal_df

def stratified_make_splits(df: pd.DataFrame, seed=42) -> Dict[str,pd.DataFrame]:
    df = df.copy().sample(frac=1, random_state=seed).reset_index(drop=True)
    y = df['label_3class'].map(CLASS_TO_ID).values
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=y, random_state=seed)
    y_temp = temp_df['label_3class'].map(CLASS_TO_ID).values
    val_df, test_df = train_test_split(temp_df, test_size=2/3, stratify=y_temp, random_state=seed)
    out = {'train': train_df.copy(), 'val': val_df.copy(), 'test': test_df.copy()}
    for k in out:
        out[k]['split'] = k
    return out

# 1) P2 splits first.
splits = try_load_p2_splits()
if splits is None:
    # 2) Complete master.
    master = try_load_allclass_master()
    if master is None:
        # 3) Rebuild from tumor table + normal images.
        tumor_df = find_tumor_gold_table()
        if tumor_df is None or len(tumor_df)==0:
            audit = {
                'reason':'No usable tumor gold table found.',
                'portion1_exists': PORTION1_INPUT.exists(),
                'portion2_exists': PORTION2_INPUT.exists(),
                'indexed_raw_like_images': len(image_by_base),
                'indexed_masks': len(mask_by_base),
            }
            (REPORT_DIR/'DATA_INPUT_AUDIT_FAILED.json').write_text(json.dumps(audit, indent=2))
            raise RuntimeError('No usable tumor table found. Check Portion-1 input and raw BTXRD dataset attachment.')
        normal_df = recover_normals_from_tables_and_images(tumor_df)
        print('Normal recovery final count:', len(normal_df))
        if len(normal_df) < 1000:
            audit = {
                'reason':'Could not recover enough normal X-rays for true 3-class training.',
                'normal_recovered': int(len(normal_df)),
                'tumor_counts': tumor_df['label_3class'].value_counts().to_dict(),
                'indexed_raw_like_image_basenames': len(image_by_base),
                'top_image_roots': root_counts.most_common(20),
                'required_action':'Attach original Kaggle dataset named BTXRD dataset, or export Portion-2 with final_internal_*.csv and actual images accessible.',
            }
            (REPORT_DIR/'DATA_INPUT_AUDIT_FAILED.json').write_text(json.dumps(audit, indent=2))
            (REPORT_DIR/'DATA_INPUT_AUDIT_FAILED.md').write_text(
                '# Data input audit failed\n\n'
                f"Recovered normal images: **{len(normal_df)}**. Need about 1,800 normal images.\n\n"
                'Attach the original **BTXRD dataset** input so the notebook can find raw radiographs.\n'
            )
            raise RuntimeError(f'Not enough normal images recovered ({len(normal_df)}). Attach the original BTXRD dataset.')
        master = pd.concat([normal_df, tumor_df], ignore_index=True, sort=False)
    splits = stratified_make_splits(master, seed=42)

# Final cleaning and saving.
for k in ['train','val','test']:
    splits[k] = add_normalized_label(splits[k])
    splits[k] = repair_paths(splits[k], verbose=False)
    splits[k] = splits[k][splits[k]['has_image'] & splits[k]['label_3class'].isin(CLASS_NAMES)].copy()
    splits[k] = splits[k].drop_duplicates(subset=['image_stem']).reset_index(drop=True)
    counts = splits[k]['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    print(f'FINAL {k} counts:', counts.to_dict(), 'n=', len(splits[k]))
    if (counts <= 0).any():
        raise RuntimeError(f'{k} split is missing a class: {counts.to_dict()}')

final_master = pd.concat(splits.values(), ignore_index=True, sort=False).reset_index(drop=True)
final_counts = final_master['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
print('FINAL MASTER COUNTS:', final_counts.to_dict(), 'n=', len(final_master))
if final_counts['normal'] < 1000 or final_counts['benign'] < 1000 or final_counts['malignant'] < 200:
    raise RuntimeError(f'Final dataset still looks incomplete: {final_counts.to_dict()}')

# Save final tables
final_master.to_csv(TABLE_DIR/'p345_master_allclass_repaired.csv', index=False)
for k,df in splits.items():
    df.to_csv(TABLE_DIR/f'p345_internal_{k}.csv', index=False)
final_counts.to_frame('count').to_csv(TABLE_DIR/'p345_final_class_counts.csv')
display(final_counts.to_frame('count'))


Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1/tables/btxrd_master_classification_all_valid.csv {'normal': 0, 'benign': 1517, 'malignant': 341}
Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1/splits/classification_all_valid_center_holdout_index.csv {'normal': 0, 'benign': 0, 'malignant': 0}
Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1/splits/classification_all_valid/classification_all_valid_val.csv {'normal': 0, 'benign': 152, 'malignant': 34}
Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1/splits/classification_all_valid/classification_all_valid_test.csv {'normal': 0, 'benign': 304, 'malignant': 68}
Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_Portion1/splits/classification_all_valid/classification_all_valid_train.csv {'normal': 0, 'benign': 1061, 'malignant': 239}
Master candidate: /kaggle/input/notebooks/koushikrudra/btxrd-p1/BTXRD_

,count
label_3class,
normal,1899
benign,1517
malignant,341


In [5]:
# ============================================================
# Cell 5 — Metadata guard and publication data visuals
# ============================================================
# Metadata is only used if it is safe. If normal rows have missing metadata but tumor rows do not,
# using metadata would create artificial leakage. This guard drops unsafe columns automatically.

SAFE_NUMERIC_HINTS = ['age']
SAFE_CATEGORICAL_HINTS = ['gender','sex','body','region','site','view','angle','location','anatomical']
DROP_METADATA_HINTS = ['path','file','image','mask','bbox','box','annotation','split','label','class','target','id','stem','source','normal_source','has_']

def choose_safe_metadata_columns(train_df, all_df):
    numeric, categorical, dropped = [], [], []
    for c in all_df.columns:
        cl = str(c).lower()
        if any(tok in cl for tok in DROP_METADATA_HINTS):
            continue
        # Candidate columns by name hint only.
        is_num_hint = any(tok == cl or tok in cl for tok in SAFE_NUMERIC_HINTS)
        is_cat_hint = any(tok == cl or tok in cl for tok in SAFE_CATEGORICAL_HINTS)
        if not (is_num_hint or is_cat_hint):
            continue
        # Coverage by class check.
        cov = train_df.groupby('label_3class')[c].apply(lambda s: s.notna().mean()).reindex(CLASS_NAMES, fill_value=0)
        if cov.min() < 0.65 or (cov.max() - cov.min()) > 0.30:
            dropped.append({'column':c, 'reason':'class-dependent missingness', **{f'coverage_{k}':float(v) for k,v in cov.items()}})
            continue
        # Cardinality sanity.
        nunique = train_df[c].nunique(dropna=True)
        if is_num_hint and pd.api.types.is_numeric_dtype(train_df[c]):
            numeric.append(c)
        elif is_num_hint:
            # Try coercion.
            coerced = pd.to_numeric(train_df[c], errors='coerce')
            if coerced.notna().mean() > 0.7:
                numeric.append(c)
            else:
                categorical.append(c)
        elif is_cat_hint and 1 < nunique <= 50:
            categorical.append(c)
    return numeric, categorical, pd.DataFrame(dropped)

META_NUM_COLS, META_CAT_COLS, dropped_meta = choose_safe_metadata_columns(splits['train'], final_master)
dropped_meta.to_csv(TABLE_DIR/'metadata_columns_dropped_leakage_guard.csv', index=False)
print('Safe numeric metadata:', META_NUM_COLS)
print('Safe categorical metadata:', META_CAT_COLS)
print('Dropped metadata columns:', len(dropped_meta))

# Fit metadata encoders.
def fit_metadata(train_df):
    stats = {}
    for c in META_NUM_COLS:
        vals = pd.to_numeric(train_df[c], errors='coerce')
        stats[c] = {'mean': float(vals.mean()), 'std': float(vals.std() if vals.std() > 1e-6 else 1.0)}
    cats = {}
    for c in META_CAT_COLS:
        vals = train_df[c].fillna('missing').astype(str)
        top = sorted(vals.unique().tolist())
        cats[c] = top
    dim = len(META_NUM_COLS) + sum(len(v) for v in cats.values())
    if dim == 0:
        dim = 1
    return {'stats':stats, 'cats':cats, 'dim':dim}

META_ENCODER = fit_metadata(splits['train'])
json.dump(META_ENCODER, open(TABLE_DIR/'metadata_encoder.json','w'), indent=2)
print('Metadata dimension:', META_ENCODER['dim'])

# Publication data visuals.
def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    return path

# fig01 split distribution
plot_df = final_master.groupby(['split','label_3class']).size().unstack(fill_value=0).reindex(columns=CLASS_NAMES, fill_value=0)
ax = plot_df.plot(kind='bar', figsize=(8,4), edgecolor='black')
ax.set_title('Corrected internal split distribution')
ax.set_xlabel('Split')
ax.set_ylabel('Number of radiographs')
ax.legend(title='Class')
savefig('fig01_corrected_split_distribution.png')

# fig02 master counts
counts = final_master['label_3class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
fig, ax = plt.subplots(figsize=(6.8,4))
ax.bar(counts.index, counts.values, edgecolor='black')
for i,v in enumerate(counts.values): ax.text(i, v+max(counts.values)*0.015, str(int(v)), ha='center')
ax.set_title('Final all-class dataset used for Portion 3/4/5')
ax.set_ylabel('Number of radiographs')
savefig('fig02_final_master_class_counts.png')

# fig03 mask coverage
mask_cov = final_master.groupby('label_3class')['has_mask'].mean().reindex(CLASS_NAMES, fill_value=0)
fig, ax = plt.subplots(figsize=(6.8,4))
ax.bar(mask_cov.index, mask_cov.values, edgecolor='black')
ax.set_ylim(0,1.05)
for i,v in enumerate(mask_cov.values): ax.text(i, v+0.03, f'{v:.2f}', ha='center')
ax.set_title('Tumor-mask availability by class')
ax.set_ylabel('Fraction with valid mask')
savefig('fig03_mask_coverage.png')

# fig04 architecture schematic
fig, ax = plt.subplots(figsize=(12,6))
ax.axis('off')
boxes = [
    (0.05,0.72,'Input X-ray\n384×384'),
    (0.27,0.72,'EfficientNet-B3\nGlobal expert'),
    (0.27,0.42,'TrustNet lesion expert\nfeature map'),
    (0.50,0.42,'Attention + mask head\nlesion/background pooling'),
    (0.50,0.72,'Global logits'),
    (0.72,0.58,'Validation-weighted\nensemble'),
    (0.72,0.30,'Temperature + offset\ncalibration'),
    (0.90,0.45,'Final output\nclass + confidence\nuncertainty + XAI'),
]
for x,y,t in boxes:
    ax.add_patch(Rectangle((x,y),0.16,0.16, fill=False, linewidth=1.8))
    ax.text(x+0.08,y+0.08,t, ha='center', va='center', fontsize=10)
arrows = [((0.21,0.80),(0.27,0.80)),((0.21,0.80),(0.27,0.50)),((0.43,0.80),(0.50,0.80)),((0.43,0.50),(0.50,0.50)),((0.66,0.80),(0.72,0.66)),((0.66,0.50),(0.72,0.62)),((0.80,0.58),(0.80,0.46)),((0.88,0.38),(0.90,0.50))]
for a,b in arrows:
    ax.annotate('', xy=b, xytext=a, arrowprops=dict(arrowstyle='->', lw=1.5))
ax.text(0.5,0.95,'BTX-TrustNet-LiteSOTA-v5: T4-efficient accuracy-protected architecture', ha='center', fontsize=14, fontweight='bold')
savefig('fig04_btx_trustnet_litesota_architecture.png')

print('Saved initial journal figures to:', FIG_DIR)


Safe numeric metadata: ['age', 'average_hash']
Safe categorical metadata: ['body_region', 'view_angle']
Dropped metadata columns: 2
Metadata dimension: 10
Saved initial journal figures to: /kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/figures


# Portion 3 — Final SOTA architecture

The model is intentionally T4-efficient. It avoids heavy external foundation models that can fail offline or exceed Kaggle runtime. The novelty is kept through the **lesion-aware TrustNet expert**, **mask/attention alignment**, **hierarchical auxiliary heads**, and **validation-only calibration/operating-point optimization**.

Main model:

```text
EfficientNet-B3 feature map
    ├── global classification expert
    └── TrustNet lesion-aware expert
            ├── attention / mask decoder
            ├── lesion-background pooling
            ├── normal-vs-tumor auxiliary head
            └── benign-vs-malignant auxiliary head
```


In [6]:
# ============================================================
# Cell 6 — Dataset, transforms, and dataloaders
# ============================================================
def encode_metadata_row(row, encoder=META_ENCODER):
    vals = []
    for c, st in encoder['stats'].items():
        x = pd.to_numeric(pd.Series([row.get(c, np.nan)]), errors='coerce').iloc[0]
        if pd.isna(x): x = st['mean']
        vals.append((float(x) - st['mean']) / st['std'])
    for c, cats in encoder['cats'].items():
        val = str(row.get(c, 'missing')) if pd.notna(row.get(c, np.nan)) else 'missing'
        vals.extend([1.0 if val == cat else 0.0 for cat in cats])
    if len(vals) == 0:
        vals = [0.0]
    return np.array(vals, dtype=np.float32)

class BTXRDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size=384, train=False, meta_encoder=None):
        self.df = df.reset_index(drop=True).copy()
        self.image_size = image_size
        self.train = train
        self.meta_encoder = meta_encoder or META_ENCODER
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    def __len__(self):
        return len(self.df)
    def load_image(self, path):
        im = Image.open(path).convert('L')
        im = ImageOps.autocontrast(im)
        im = im.convert('RGB')
        return im
    def load_mask(self, path):
        if isinstance(path, str) and Path(path).exists():
            try:
                m = Image.open(path).convert('L')
                return m
            except Exception:
                pass
        return Image.new('L', (self.image_size, self.image_size), 0)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = CLASS_TO_ID[row['label_3class']]
        im = self.load_image(row['image_path'])
        mask = self.load_mask(row.get('mask_path', None)) if y > 0 else Image.new('L', im.size, 0)
        # Resize first. Keep transforms safe and mask-aligned.
        im = TF.resize(im, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.NEAREST)
        if self.train:
            # Conservative radiograph-safe augmentation.
            angle = random.uniform(-7, 7)
            im = TF.rotate(im, angle, interpolation=TF.InterpolationMode.BILINEAR, fill=0)
            mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.35:
                im = ImageEnhance.Contrast(im).enhance(random.uniform(0.85, 1.20))
            if random.random() < 0.35:
                im = ImageEnhance.Brightness(im).enhance(random.uniform(0.90, 1.10))
            # Keep horizontal flip disabled by default; can be turned on after anatomical review.
        x = TF.to_tensor(im)
        x = (x - self.mean) / self.std
        m = TF.to_tensor(mask)
        m = (m > 0.5).float()
        meta = torch.tensor(encode_metadata_row(row, self.meta_encoder), dtype=torch.float32)
        return {
            'image': x,
            'mask': m,
            'label': torch.tensor(y, dtype=torch.long),
            'meta': meta,
            'index': torch.tensor(idx, dtype=torch.long),
        }

def make_loader(df, train=False, batch_size=None):
    batch_size = batch_size or CFG['batch_size']
    ds = BTXRDDataset(df, image_size=CFG['image_size'], train=train, meta_encoder=META_ENCODER)
    sampler = None
    shuffle = train
    if train and CFG['use_weighted_sampler']:
        labels = df['label_3class'].map(CLASS_TO_ID).values
        counts = np.bincount(labels, minlength=3)
        weights = 1.0 / np.sqrt(np.maximum(counts, 1))
        sample_w = weights[labels]
        sampler = WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double), num_samples=len(sample_w), replacement=True)
        shuffle = False
    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
                        num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(), drop_last=False)
    return loader

train_loader = make_loader(splits['train'], train=True)
val_loader = make_loader(splits['val'], train=False, batch_size=CFG['batch_size']*2)
test_loader = make_loader(splits['test'], train=False, batch_size=CFG['batch_size']*2)
print('Batches:', len(train_loader), len(val_loader), len(test_loader))


Batches: 165 12 24


In [7]:
# ============================================================
# Cell 7 — Model definitions: GlobalExpert and TrustNet-LiteSOTA-v5
# ============================================================
def build_efficientnet_b3_features(pretrained=True):
    if efficientnet_b3 is None:
        raise RuntimeError('torchvision efficientnet_b3 is not available in this environment.')
    weights = None
    if pretrained and EfficientNet_B3_Weights is not None and ALLOW_TORCHVISION_WEIGHT_DOWNLOAD:
        try:
            weights = EfficientNet_B3_Weights.IMAGENET1K_V1
            model = efficientnet_b3(weights=weights)
            print('Loaded torchvision EfficientNet-B3 ImageNet weights.')
        except Exception as e:
            print('ImageNet weight load failed; using random init. Reason:', str(e)[:180])
            model = efficientnet_b3(weights=None)
    else:
        model = efficientnet_b3(weights=None)
    return model.features, 1536

class GlobalExpert(nn.Module):
    def __init__(self, meta_dim=1, n_classes=3, dropout=0.35, pretrained=True):
        super().__init__()
        self.features, ch = build_efficientnet_b3_features(pretrained=pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.meta = nn.Sequential(nn.Linear(meta_dim, 32), nn.ReLU(), nn.Dropout(0.15)) if meta_dim > 1 else None
        in_dim = ch + (32 if meta_dim > 1 else 0)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, n_classes))
    def forward(self, x, meta=None, return_features=False):
        f = self.features(x)
        g = self.pool(f).flatten(1)
        z = g
        if self.meta is not None and meta is not None:
            z = torch.cat([z, self.meta(meta)], dim=1)
        logits = self.classifier(z)
        if return_features:
            return logits, g
        return logits

class TrustNetLiteSOTA(nn.Module):
    def __init__(self, meta_dim=1, n_classes=3, dropout=0.35, pretrained=True):
        super().__init__()
        self.features, ch = build_efficientnet_b3_features(pretrained=pretrained)
        self.attn_head = nn.Sequential(
            nn.Conv2d(ch, 256, 1), nn.BatchNorm2d(256), nn.SiLU(inplace=True),
            nn.Conv2d(256, 1, 1)
        )
        self.mask_head = nn.Sequential(
            nn.Conv2d(ch, 256, 3, padding=1), nn.BatchNorm2d(256), nn.SiLU(inplace=True),
            nn.Conv2d(256, 1, 1)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.meta = nn.Sequential(nn.Linear(meta_dim, 32), nn.ReLU(), nn.Dropout(0.15)) if meta_dim > 1 else None
        in_dim = ch * 3 + (32 if meta_dim > 1 else 0)
        self.fusion = nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(dropout), nn.Linear(in_dim, 768), nn.SiLU(), nn.Dropout(dropout/2))
        self.main_head = nn.Linear(768, n_classes)
        self.nt_head = nn.Linear(768, 2)   # normal vs tumor
        self.bm_head = nn.Linear(768, 2)   # benign vs malignant for tumor cases
    def forward(self, x, meta=None, return_features=False):
        f = self.features(x)
        att_logit_small = self.attn_head(f)
        mask_logit_small = self.mask_head(f)
        att = torch.sigmoid(att_logit_small)
        eps = 1e-6
        global_feat = self.pool(f).flatten(1)
        lesion_feat = (f * att).sum(dim=(2,3)) / (att.sum(dim=(2,3)) + eps)
        bg = 1.0 - att
        bg_feat = (f * bg).sum(dim=(2,3)) / (bg.sum(dim=(2,3)) + eps)
        z = torch.cat([global_feat, lesion_feat, bg_feat], dim=1)
        if self.meta is not None and meta is not None:
            z = torch.cat([z, self.meta(meta)], dim=1)
        h = self.fusion(z)
        out = {
            'logits': self.main_head(h),
            'nt_logits': self.nt_head(h),
            'bm_logits': self.bm_head(h),
            'attn_small': att_logit_small,
            'mask_small': mask_logit_small,
            'features': global_feat,
        }
        if return_features:
            return out
        return out

# Losses
class FocalCE(nn.Module):
    def __init__(self, weight=None, gamma=1.5):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction='none')
        pt = torch.exp(-ce).clamp(1e-6, 1.0)
        return (((1-pt)**self.gamma) * ce).mean()

def dice_loss_from_logits(logits, target, eps=1e-6):
    probs = torch.sigmoid(logits)
    target = target.float()
    inter = (probs * target).sum(dim=(1,2,3))
    den = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return (1 - (2*inter + eps)/(den + eps)).mean()

def compute_class_weights(df):
    y = df['label_3class'].map(CLASS_TO_ID).values
    counts = np.bincount(y, minlength=3).astype(float)
    w = 1.0 / np.sqrt(np.maximum(counts, 1))
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

CLASS_WEIGHTS = compute_class_weights(splits['train'])
print('Class weights:', CLASS_WEIGHTS.detach().cpu().numpy())


Class weights: [0.67004675 0.7499113  1.580042  ]


# Portion 4 — T4-efficient best-accuracy training plan

The training is staged and accuracy-protected:

1. **GlobalExpert**: EfficientNet-B3 classifier. This protects accuracy and gives a strong fallback.
2. **TrustNetLiteSOTA**: lesion-attention + mask decoder + hierarchical auxiliary heads.
3. **Validation-weighted ensemble**: chooses how much to trust the global expert vs lesion expert on the validation set only.
4. **Temperature scaling and offset tuning**: calibrated and validation-only.
5. **Selective referral**: not the primary blind-test result; used as clinical operating-point analysis.


In [8]:
# ============================================================
# Cell 8 — Metrics, ECE, prediction helpers
# ============================================================
def softmax_np(logits):
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)

def ece_score(y_true, probs, n_bins=15):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(acc[mask].mean() - conf[mask].mean())
    return float(ece)

def safe_auc_auprc(y_true, probs):
    y_bin = label_binarize(y_true, classes=[0,1,2])
    out = {}
    try: out['macro_auroc'] = roc_auc_score(y_bin, probs, average='macro', multi_class='ovr')
    except Exception: out['macro_auroc'] = np.nan
    try: out['weighted_auroc'] = roc_auc_score(y_bin, probs, average='weighted', multi_class='ovr')
    except Exception: out['weighted_auroc'] = np.nan
    aps=[]
    for i,c in enumerate(CLASS_NAMES):
        try: out[f'{c}_auroc'] = roc_auc_score(y_bin[:,i], probs[:,i])
        except Exception: out[f'{c}_auroc'] = np.nan
        try: out[f'{c}_auprc'] = average_precision_score(y_bin[:,i], probs[:,i]); aps.append(out[f'{c}_auprc'])
        except Exception: out[f'{c}_auprc'] = np.nan
    out['macro_auprc'] = float(np.nanmean(aps)) if len(aps) else np.nan
    return out

def all_metrics(y_true, probs, prefix=''):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs)
    pred = probs.argmax(axis=1)
    cm = confusion_matrix(y_true, pred, labels=[0,1,2])
    m = {}
    m[prefix+'accuracy'] = accuracy_score(y_true, pred)
    m[prefix+'balanced_accuracy'] = balanced_accuracy_score(y_true, pred)
    m[prefix+'macro_f1'] = f1_score(y_true, pred, average='macro', zero_division=0)
    m[prefix+'weighted_f1'] = f1_score(y_true, pred, average='weighted', zero_division=0)
    m[prefix+'macro_precision'] = precision_score(y_true, pred, average='macro', zero_division=0)
    m[prefix+'macro_recall'] = recall_score(y_true, pred, average='macro', zero_division=0)
    m[prefix+'cohen_kappa'] = cohen_kappa_score(y_true, pred)
    m[prefix+'mcc'] = matthews_corrcoef(y_true, pred)
    top2 = np.argsort(-probs, axis=1)[:,:2]
    m[prefix+'top2_accuracy'] = np.mean([y_true[i] in top2[i] for i in range(len(y_true))])
    try: m[prefix+'nll'] = log_loss(y_true, probs, labels=[0,1,2])
    except Exception: m[prefix+'nll'] = np.nan
    try:
        y_bin = label_binarize(y_true, classes=[0,1,2])
        m[prefix+'brier_score'] = np.mean(np.sum((probs-y_bin)**2, axis=1))
    except Exception: m[prefix+'brier_score'] = np.nan
    m[prefix+'ece'] = ece_score(y_true, probs)
    aucs = safe_auc_auprc(y_true, probs)
    for k,v in aucs.items(): m[prefix+k] = v
    for i,c in enumerate(CLASS_NAMES):
        tp = cm[i,i]
        fn = cm[i,:].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp+fn+1e-12)
        spec = tn / (tn+fp+1e-12)
        prec = tp / (tp+fp+1e-12)
        f1 = 2*prec*sens/(prec+sens+1e-12)
        npv = tn / (tn+fn+1e-12)
        m[prefix+f'{c}_sensitivity'] = sens
        m[prefix+f'{c}_specificity'] = spec
        m[prefix+f'{c}_precision'] = prec
        m[prefix+f'{c}_f1'] = f1
        m[prefix+f'{c}_npv'] = npv
        m[prefix+f'{c}_fnr'] = fn/(tp+fn+1e-12)
        m[prefix+f'{c}_fpr'] = fp/(fp+tn+1e-12)
    return m

def objective_from_metrics(m):
    return (0.32*m.get('macro_f1',0) + 0.25*m.get('malignant_sensitivity',0) +
            0.22*m.get('balanced_accuracy',0) + 0.16*m.get('malignant_auprc',0) -
            0.05*m.get('ece',0))

@torch.no_grad()
def predict_model(model, loader, model_type='trust'):
    model.eval()
    logits_all, y_all, idx_all, feat_all = [], [], [], []
    attn_all, mask_all = [], []
    for batch in loader:
        x = batch['image'].to(device, non_blocking=True)
        meta = batch['meta'].to(device, non_blocking=True)
        y = batch['label'].cpu().numpy()
        if model_type == 'global':
            logits, feats = model(x, meta, return_features=True)
            out_logits = logits
            feat_all.append(feats.detach().cpu().numpy())
        else:
            out = model(x, meta, return_features=True)
            out_logits = out['logits']
            feat_all.append(out['features'].detach().cpu().numpy())
            att = F.interpolate(out['attn_small'], size=x.shape[-2:], mode='bilinear', align_corners=False)
            ms = F.interpolate(out['mask_small'], size=x.shape[-2:], mode='bilinear', align_corners=False)
            attn_all.append(torch.sigmoid(att).detach().cpu().numpy())
            mask_all.append(torch.sigmoid(ms).detach().cpu().numpy())
        logits_all.append(out_logits.detach().cpu().numpy())
        y_all.append(y)
        idx_all.append(batch['index'].cpu().numpy())
    res = {
        'logits': np.concatenate(logits_all),
        'y': np.concatenate(y_all),
        'indices': np.concatenate(idx_all),
        'features': np.concatenate(feat_all) if feat_all else None,
    }
    if attn_all: res['attn'] = np.concatenate(attn_all)
    if mask_all: res['mask_pred'] = np.concatenate(mask_all)
    return res


In [9]:
# ============================================================
# Cell 9 — Training loops
# ============================================================
def set_backbone_trainable(model, trainable: bool):
    if hasattr(model, 'features'):
        for p in model.features.parameters():
            p.requires_grad = trainable

def train_one_epoch_global(model, loader, optimizer, scaler, loss_fn):
    model.train(); total=0; n=0
    for batch in loader:
        x = batch['image'].to(device, non_blocking=True)
        y = batch['label'].to(device, non_blocking=True)
        meta = batch['meta'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CFG['amp'] and device.type=='cuda'):
            logits = model(x, meta)
            loss = loss_fn(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()*len(y); n += len(y)
    return total/max(n,1)

def train_one_epoch_trust(model, loader, optimizer, scaler, main_loss_fn):
    model.train(); total=0; n=0
    for batch in loader:
        x = batch['image'].to(device, non_blocking=True)
        y = batch['label'].to(device, non_blocking=True)
        meta = batch['meta'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)
        tumor = y > 0
        nt = (y > 0).long()
        bm = (y - 1).clamp(min=0, max=1)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CFG['amp'] and device.type=='cuda'):
            out = model(x, meta)
            loss_main = main_loss_fn(out['logits'], y)
            loss_nt = F.cross_entropy(out['nt_logits'], nt)
            loss_bm = torch.tensor(0.0, device=device)
            if tumor.any():
                loss_bm = F.cross_entropy(out['bm_logits'][tumor], bm[tumor])
            loss_mask = torch.tensor(0.0, device=device)
            if tumor.any() and mask[tumor].sum() > 0:
                mask_small = F.interpolate(mask, size=out['mask_small'].shape[-2:], mode='nearest')
                loss_bce = F.binary_cross_entropy_with_logits(out['mask_small'][tumor], mask_small[tumor])
                loss_dice = dice_loss_from_logits(out['mask_small'][tumor], mask_small[tumor])
                att_loss = F.binary_cross_entropy_with_logits(out['attn_small'][tumor], mask_small[tumor])
                loss_mask = loss_bce + loss_dice + 0.25*att_loss
            loss = loss_main + 0.20*loss_nt + 0.30*loss_bm + 0.18*loss_mask
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()*len(y); n += len(y)
    return total/max(n,1)

def validate_model(model, loader, model_type='trust'):
    res = predict_model(model, loader, model_type=model_type)
    probs = softmax_np(res['logits'])
    m = all_metrics(res['y'], probs)
    return m, res

def find_p2_checkpoint():
    ckpts = []
    for root in [PORTION2_INPUT] + P2_ROOTS + [Path('/kaggle/input')]:
        if root.exists():
            try:
                for p in root.rglob('*.pt'):
                    name = p.name.lower()
                    if any(tok in name for tok in ['efficientnet','b3','best','baseline','model']):
                        ckpts.append(p)
                for p in root.rglob('*.pth'):
                    name = p.name.lower()
                    if any(tok in name for tok in ['efficientnet','b3','best','baseline','model']):
                        ckpts.append(p)
            except Exception:
                pass
    ckpts = sorted(set(ckpts), key=lambda p: (0 if 'btxrd-p2' in pstr(p).lower() else 1, len(pstr(p))))
    return ckpts[0] if ckpts else None

def try_load_checkpoint(model, ckpt_path):
    """Robust checkpoint loader. Loads only shape-matched tensors and handles common prefixes."""
    if ckpt_path is None or not Path(ckpt_path).exists():
        return False
    try:
        obj = torch.load(ckpt_path, map_location='cpu')
        sd = obj.get('state_dict', obj.get('model_state_dict', obj.get('model', obj))) if isinstance(obj, dict) else obj
        if not isinstance(sd, dict):
            return False
        target = model.state_dict()
        replacements = [
            ('module.', ''), ('model.', ''), ('net.', ''),
            ('backbone.features.', 'features.'), ('model.features.', 'features.'),
            ('encoder.features.', 'features.'), ('base.features.', 'features.'),
            ('backbone.', 'features.'),
        ]
        filtered = {}
        for k, v in sd.items():
            candidates = [k]
            kk = k
            for a,b in replacements:
                if kk.startswith(a):
                    candidates.append(b + kk[len(a):])
                if a in kk:
                    candidates.append(kk.replace(a,b))
            for cand in list(dict.fromkeys(candidates)):
                if cand in target and tuple(target[cand].shape) == tuple(v.shape):
                    filtered[cand] = v
                    break
        if len(filtered) == 0:
            print('Checkpoint found but no matching tensors:', ckpt_path)
            return False
        target.update(filtered)
        model.load_state_dict(target, strict=False)
        print(f'Loaded {len(filtered)}/{len(target)} tensors from checkpoint:', ckpt_path)
        return True
    except Exception as e:
        print('Checkpoint load skipped:', ckpt_path, 'reason:', str(e)[:220])
        return False

p2_ckpt = find_p2_checkpoint()
print('P2 checkpoint candidate:', p2_ckpt)


P2 checkpoint candidate: None


In [10]:
# ============================================================
# Cell 10 — Train GlobalExpert + TrustNetLiteSOTA seeds
# ============================================================
training_history = []
seed_outputs = []
meta_dim = META_ENCODER['dim']

# Main loss
main_loss_fn = FocalCE(weight=CLASS_WEIGHTS, gamma=1.4)

for seed in CFG['seeds']:
    print('\n' + '='*80)
    print('Training seed:', seed)
    print('='*80)
    seed_everything(seed)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG['amp'] and device.type=='cuda')

    # ---------------------------
    # GlobalExpert
    # ---------------------------
    global_model = GlobalExpert(meta_dim=meta_dim, pretrained=True).to(device)
    loaded_global = False
    if p2_ckpt is not None:
        loaded_global = try_load_checkpoint(global_model, p2_ckpt)
    if not loaded_global and CFG['train_global_if_no_p2']:
        print('Training GlobalExpert from available initialization.')
        best_score = -1e9; best_state = None; wait = 0
        opt = torch.optim.AdamW(global_model.parameters(), lr=2e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, CFG['epochs_global']))
        for epoch in range(1, CFG['epochs_global']+1):
            set_backbone_trainable(global_model, epoch > CFG['freeze_epochs'])
            loss = train_one_epoch_global(global_model, train_loader, opt, scaler, main_loss_fn)
            sched.step()
            vm, _ = validate_model(global_model, val_loader, model_type='global')
            score = objective_from_metrics(vm)
            training_history.append({'seed':seed,'model':'GlobalExpert','epoch':epoch,'loss':loss,'score':score, **vm})
            print(f'[Global seed {seed} epoch {epoch}] loss={loss:.4f} score={score:.4f} macroF1={vm["macro_f1"]:.4f} malSens={vm["malignant_sensitivity"]:.4f} balAcc={vm["balanced_accuracy"]:.4f}')
            if score > best_score:
                best_score = score; best_state = {k:v.detach().cpu() for k,v in global_model.state_dict().items()}; wait = 0
            else:
                wait += 1
                if wait >= CFG['early_stop']:
                    print('GlobalExpert early stopping.')
                    break
        if best_state is not None:
            global_model.load_state_dict(best_state)
    else:
        print('Using loaded GlobalExpert checkpoint.')

    global_path = MODEL_DIR / f'global_expert_seed{seed}.pt'
    torch.save({'state_dict': global_model.state_dict(), 'seed':seed, 'meta_encoder':META_ENCODER}, global_path)

    # ---------------------------
    # TrustNetLiteSOTA
    # ---------------------------
    trust_model = TrustNetLiteSOTA(meta_dim=meta_dim, pretrained=True).to(device)
    # Warm-start TrustNet feature extractor from global model feature extractor.
    try:
        trust_model.features.load_state_dict(global_model.features.state_dict(), strict=False)
        print('TrustNet feature extractor warm-started from GlobalExpert.')
    except Exception as e:
        print('TrustNet warm-start skipped:', e)

    if CFG['train_trustnet']:
        best_score = -1e9; best_state = None; wait = 0
        opt = torch.optim.AdamW(trust_model.parameters(), lr=2e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, CFG['epochs_trust']))
        for epoch in range(1, CFG['epochs_trust']+1):
            set_backbone_trainable(trust_model, epoch > CFG['freeze_epochs'])
            loss = train_one_epoch_trust(trust_model, train_loader, opt, scaler, main_loss_fn)
            sched.step()
            vm, _ = validate_model(trust_model, val_loader, model_type='trust')
            score = objective_from_metrics(vm)
            training_history.append({'seed':seed,'model':'TrustNetLiteSOTA','epoch':epoch,'loss':loss,'score':score, **vm})
            print(f'[Trust seed {seed} epoch {epoch}] loss={loss:.4f} score={score:.4f} macroF1={vm["macro_f1"]:.4f} malSens={vm["malignant_sensitivity"]:.4f} balAcc={vm["balanced_accuracy"]:.4f} ece={vm["ece"]:.4f}')
            if score > best_score:
                best_score = score; best_state = {k:v.detach().cpu() for k,v in trust_model.state_dict().items()}; wait = 0
            else:
                wait += 1
                if wait >= CFG['early_stop']:
                    print('TrustNet early stopping.')
                    break
        if best_state is not None:
            trust_model.load_state_dict(best_state)

    trust_path = MODEL_DIR / f'trustnet_litesota_seed{seed}.pt'
    torch.save({'state_dict': trust_model.state_dict(), 'seed':seed, 'meta_encoder':META_ENCODER}, trust_path)

    # Collect predictions.
    g_val = predict_model(global_model, val_loader, 'global')
    g_test = predict_model(global_model, test_loader, 'global')
    t_val = predict_model(trust_model, val_loader, 'trust')
    t_test = predict_model(trust_model, test_loader, 'trust')
    seed_outputs.append({
        'seed': seed,
        'global_val_logits': g_val['logits'], 'global_test_logits': g_test['logits'],
        'trust_val_logits': t_val['logits'], 'trust_test_logits': t_test['logits'],
        'trust_test_attn': t_test.get('attn'), 'trust_test_mask_pred': t_test.get('mask_pred'),
        'trust_test_features': t_test.get('features'),
        'val_y': g_val['y'], 'test_y': g_test['y'], 'test_indices': g_test['indices'],
    })
    del global_model, trust_model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

hist_df = pd.DataFrame(training_history)
hist_df.to_csv(TABLE_DIR/'p4_training_history.csv', index=False)
display(hist_df.tail())



Training seed: 42
Training GlobalExpert from available initialization.
[Global seed 42 epoch 1] loss=0.5190 score=0.3429 macroF1=0.0553 malSens=1.0000 balAcc=0.3333
[Global seed 42 epoch 2] loss=0.5192 score=0.4309 macroF1=0.2969 malSens=0.8824 balAcc=0.4472
[Global seed 42 epoch 3] loss=0.4692 score=0.4931 macroF1=0.4309 malSens=0.7647 balAcc=0.5334
[Global seed 42 epoch 4] loss=0.4498 score=0.5083 macroF1=0.3922 malSens=0.9118 balAcc=0.5302
TrustNet feature extractor warm-started from GlobalExpert.
[Trust seed 42 epoch 1] loss=1.1445 score=0.4585 macroF1=0.4829 malSens=0.5882 balAcc=0.5465 ece=0.0593
[Trust seed 42 epoch 2] loss=1.1011 score=0.4257 macroF1=0.3328 malSens=0.7353 balAcc=0.4714 ece=0.1794
[Trust seed 42 epoch 3] loss=1.0288 score=0.4625 macroF1=0.3109 malSens=0.9706 balAcc=0.4819 ece=0.4257
[Trust seed 42 epoch 4] loss=0.9665 score=0.5129 macroF1=0.4547 malSens=0.7647 balAcc=0.5619 ece=0.0703
[Trust seed 42 epoch 5] loss=0.8875 score=0.4956 macroF1=0.4558 malSens=0.764

,seed,model,epoch,loss,score,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,...,benign_npv,benign_fnr,benign_fpr,malignant_sensitivity,malignant_specificity,malignant_precision,malignant_f1,malignant_npv,malignant_fnr,malignant_fpr
23,909,TrustNetLiteSOTA,6,0.864026,0.586931,0.555851,0.640170,0.524987,0.582832,0.550134,...,0.698473,0.519737,0.183036,0.882353,0.725146,0.241935,0.379747,0.984127,0.117647,0.274854
24,909,TrustNetLiteSOTA,7,0.852078,0.595621,0.622340,0.667492,0.572667,0.642780,0.598469,...,0.717314,0.526316,0.093750,0.823529,0.771930,0.264151,0.400000,0.977778,0.176471,0.228070
25,909,TrustNetLiteSOTA,8,0.813872,0.592279,0.558511,0.636507,0.533259,0.574319,0.535280,...,0.689362,0.480263,0.276786,0.852941,0.792398,0.290000,0.432836,0.981884,0.147059,0.207602
26,909,TrustNetLiteSOTA,9,0.774973,0.585709,0.574468,0.635036,0.535604,0.595539,0.546341,...,0.684615,0.539474,0.205357,0.823529,0.766082,0.259259,0.394366,0.977612,0.176471,0.233918
27,909,TrustNetLiteSOTA,10,0.736416,0.605737,0.601064,0.653896,0.565357,0.610391,0.557159,...,0.693798,0.519737,0.200893,0.823529,0.827485,0.321839,0.462810,0.979239,0.176471,0.172515


In [11]:
# ============================================================
# Cell 11 — Ensemble, calibration, validation-only optimization
# ============================================================
def logits_to_probs(logits, T=1.0, offset=None):
    z = np.asarray(logits, dtype=np.float64) / max(T, 1e-6)
    if offset is not None:
        z = z + np.asarray(offset).reshape(1,-1)
    return softmax_np(z)

def optimize_temperature(logits, y):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    logT = torch.tensor([0.0], requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
    def closure():
        opt.zero_grad()
        T = torch.exp(logT).clamp(0.05, 10.0)
        loss = F.cross_entropy(logits_t / T, y_t)
        loss.backward()
        return loss
    try:
        opt.step(closure)
    except Exception:
        pass
    return float(torch.exp(logT).detach().clamp(0.05, 10.0).item())

def optimize_ensemble_weight(g_logits, t_logits, y):
    best = {'w':0.5, 'score':-1e9, 'metrics':None}
    for w in np.linspace(0,1,21):
        logits = w*t_logits + (1-w)*g_logits
        probs = logits_to_probs(logits)
        m = all_metrics(y, probs)
        score = objective_from_metrics(m)
        if score > best['score']:
            best = {'w':float(w), 'score':score, 'metrics':m}
    return best

def optimize_logit_offset(logits, y):
    # Small validation-only grid. Offsets sum does not matter; use normal/benign/malignant offsets.
    best = {'offset':np.zeros(3), 'score':-1e9, 'metrics':None}
    grid = np.linspace(-0.8, 0.8, 9)
    for ob in grid:
        for om in grid:
            offset = np.array([0.0, ob, om])
            probs = logits_to_probs(logits, offset=offset)
            m = all_metrics(y, probs)
            score = objective_from_metrics(m)
            if score > best['score']:
                best = {'offset':offset, 'score':score, 'metrics':m}
    return best

# Stack seed logits.
val_y = seed_outputs[0]['val_y']
test_y = seed_outputs[0]['test_y']
# Average global/trust logits across seeds.
g_val_logits = np.mean([o['global_val_logits'] for o in seed_outputs], axis=0)
g_test_logits = np.mean([o['global_test_logits'] for o in seed_outputs], axis=0)
t_val_logits = np.mean([o['trust_val_logits'] for o in seed_outputs], axis=0)
t_test_logits = np.mean([o['trust_test_logits'] for o in seed_outputs], axis=0)

ens = optimize_ensemble_weight(g_val_logits, t_val_logits, val_y)
w = ens['w']
print('Best validation TrustNet weight:', w, 'score:', ens['score'])
val_logits_ens = w*t_val_logits + (1-w)*g_val_logits
test_logits_ens = w*t_test_logits + (1-w)*g_test_logits

T = optimize_temperature(val_logits_ens, val_y)
print('Optimized temperature:', T)

offset_opt = optimize_logit_offset(val_logits_ens / T, val_y)
offset = offset_opt['offset']
print('Optimized logit offset:', offset, 'score:', offset_opt['score'])

# Primary and optimized test probabilities.
val_probs_primary = logits_to_probs(val_logits_ens, T=T)
test_probs_primary = logits_to_probs(test_logits_ens, T=T)
val_probs_offset = logits_to_probs(val_logits_ens, T=T, offset=offset)
test_probs_offset = logits_to_probs(test_logits_ens, T=T, offset=offset)

primary_metrics = all_metrics(test_y, test_probs_primary)
offset_metrics = all_metrics(test_y, test_probs_offset)
print('Primary calibrated metrics:')
display(pd.DataFrame([primary_metrics]).T.rename(columns={0:'value'}).head(35))
print('Offset operating-point metrics:')
display(pd.DataFrame([offset_metrics]).T.rename(columns={0:'value'}).head(35))

pd.DataFrame([{'trustnet_weight':w, 'temperature':T, 'offset_normal':offset[0], 'offset_benign':offset[1], 'offset_malignant':offset[2]}]).to_csv(TABLE_DIR/'p4_ensemble_calibration_settings.csv', index=False)
pd.DataFrame([primary_metrics], index=['primary_calibrated']).to_csv(TABLE_DIR/'p5_primary_calibrated_metrics.csv')
pd.DataFrame([offset_metrics], index=['offset_operating_point']).to_csv(TABLE_DIR/'p5_offset_operating_point_metrics.csv')

np.save(ARR_DIR/'test_probs_primary_calibrated.npy', test_probs_primary)
np.save(ARR_DIR/'test_probs_offset_operating.npy', test_probs_offset)
np.save(ARR_DIR/'test_labels.npy', test_y)
np.save(ARR_DIR/'val_probs_primary_calibrated.npy', val_probs_primary)
np.save(ARR_DIR/'val_labels.npy', val_y)


Best validation TrustNet weight: 0.8500000000000001 score: 0.6317556712883075
Optimized temperature: 0.8148447871208191
Optimized logit offset: [0.  0.  0.4] score: 0.6446266689348064
Primary calibrated metrics:


,value
accuracy,0.591755
balanced_accuracy,0.619659
macro_f1,0.544002
weighted_f1,0.596549
macro_precision,0.542127
macro_recall,0.619659
cohen_kappa,0.334908
mcc,0.344809
top2_accuracy,0.886968
nll,0.870482


Offset operating-point metrics:


,value
accuracy,0.555851
balanced_accuracy,0.607030
macro_f1,0.504733
weighted_f1,0.567298
macro_precision,0.532203
macro_recall,0.607030
cohen_kappa,0.307398
mcc,0.328160
top2_accuracy,0.867021
nll,0.944406


# Portion 5 — Journal-level evaluation and visuals

This section produces the final paper package:

- **30+ metrics**
- **bootstrap 95% confidence intervals**
- **selective referral/risk-coverage**
- **threshold and decision-curve analysis**
- **XAI attention-mask audit**
- **feature-space visualization**
- **35+ journal-ready figures at 400 DPI**


In [12]:
# ============================================================
# Cell 12 — Portion 5: 30+ metrics and bootstrap confidence intervals
# ============================================================
FINAL_PROBS = test_probs_offset  # Use optimized operating point for best accuracy table.
PRIMARY_PROBS = test_probs_primary
FINAL_NAME = 'BTX-TrustNet-LiteSOTA-v5-offset'
PRIMARY_NAME = 'BTX-TrustNet-LiteSOTA-v5-primary-calibrated'

primary_m = all_metrics(test_y, PRIMARY_PROBS)
final_m = all_metrics(test_y, FINAL_PROBS)
metrics_df = pd.DataFrame([primary_m, final_m], index=[PRIMARY_NAME, FINAL_NAME])
metrics_df.to_csv(TABLE_DIR/'table01_final_30plus_metrics.csv')
display(metrics_df.T)

# Bootstrap CI for key metrics.
KEY_METRICS = ['accuracy','balanced_accuracy','macro_f1','weighted_f1','malignant_sensitivity','malignant_specificity','malignant_auprc','macro_auroc','macro_auprc','ece','nll','brier_score','mcc','cohen_kappa']

def bootstrap_ci(y, probs, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    n = len(y)
    for metric in KEY_METRICS:
        vals = []
        for _ in range(n_boot):
            idx = rng.integers(0, n, size=n)
            # require at least two classes in bootstrap for stable metrics
            try:
                mm = all_metrics(y[idx], probs[idx])
                vals.append(mm.get(metric, np.nan))
            except Exception:
                vals.append(np.nan)
        vals = np.array(vals, dtype=float)
        rows.append({
            'metric':metric,
            'estimate':all_metrics(y, probs).get(metric, np.nan),
            'ci_low':np.nanpercentile(vals, 2.5),
            'ci_high':np.nanpercentile(vals, 97.5),
            'bootstrap_n':n_boot,
        })
    return pd.DataFrame(rows)

ci_df = bootstrap_ci(test_y, FINAL_PROBS, n_boot=CFG['bootstrap_n'])
ci_df.to_csv(TABLE_DIR/'table02_bootstrap_95ci.csv', index=False)
display(ci_df)


,BTX-TrustNet-LiteSOTA-v5-primary-calibrated,BTX-TrustNet-LiteSOTA-v5-offset
accuracy,0.591755,0.555851
balanced_accuracy,0.619659,0.607030
macro_f1,0.544002,0.504733
weighted_f1,0.596549,0.567298
macro_precision,0.542127,0.532203
macro_recall,0.619659,0.607030
cohen_kappa,0.334908,0.307398
mcc,0.344809,0.328160
top2_accuracy,0.886968,0.867021
nll,0.870482,0.944406


,metric,estimate,ci_low,ci_high,bootstrap_n
0,accuracy,0.555851,0.525233,0.594415,400
1,balanced_accuracy,0.607030,0.568426,0.649158,400
2,macro_f1,0.504733,0.469266,0.540044,400
3,weighted_f1,0.567298,0.527757,0.606491,400
4,malignant_sensitivity,0.794118,0.684872,0.906279,400
5,malignant_specificity,0.748538,0.716569,0.780218,400
6,malignant_auprc,0.462783,0.351385,0.591346,400
7,macro_auroc,0.758198,0.735645,0.786996,400
8,macro_auprc,0.603356,0.562976,0.654067,400
9,ece,0.072101,0.066271,0.120042,400


In [13]:
# ============================================================
# Cell 13 — Portion 5: selective referral, threshold sweeps, decision curves
# ============================================================
def selective_referral_table(y, probs, coverages=[1.0,0.95,0.90,0.85,0.80,0.75,0.70,0.60]):
    conf = probs.max(axis=1)
    rows=[]
    for cov in coverages:
        k = max(1, int(round(len(y)*cov)))
        keep_idx = np.argsort(-conf)[:k]
        m = all_metrics(y[keep_idx], probs[keep_idx])
        rows.append({
            'coverage':cov,
            'retained_n':k,
            'referred_n':len(y)-k,
            'accuracy':m['accuracy'],
            'balanced_accuracy':m['balanced_accuracy'],
            'macro_f1':m['macro_f1'],
            'malignant_sensitivity':m['malignant_sensitivity'],
            'ece':m['ece'],
        })
    return pd.DataFrame(rows)

selective_df = selective_referral_table(test_y, FINAL_PROBS)
selective_df.to_csv(TABLE_DIR/'table03_selective_referral.csv', index=False)
display(selective_df)

# Malignant threshold sweep with normal/benign default argmax.
def malignant_threshold_sweep(y, probs):
    rows=[]
    for th in np.linspace(0.05,0.95,91):
        pred = probs.argmax(axis=1)
        pred[probs[:,2] >= th] = 2
        cm = confusion_matrix(y, pred, labels=[0,1,2])
        tmp_probs = np.eye(3)[pred] * 0.98 + 0.01
        m = all_metrics(y, tmp_probs)
        rows.append({'threshold':th, **{k:m[k] for k in ['accuracy','balanced_accuracy','macro_f1','malignant_sensitivity','malignant_specificity','malignant_precision','malignant_f1']}})
    return pd.DataFrame(rows)

thr_df = malignant_threshold_sweep(test_y.copy(), FINAL_PROBS.copy())
thr_df.to_csv(TABLE_DIR/'table04_malignant_threshold_sweep.csv', index=False)

# Decision curve analysis for malignant one-vs-rest.
def decision_curve(y, probs):
    y_bin = (y==2).astype(int)
    p = probs[:,2]
    rows=[]; n=len(y)
    prevalence = y_bin.mean()
    for pt in np.linspace(0.05,0.80,76):
        pred = p >= pt
        tp = ((pred==1)&(y_bin==1)).sum()
        fp = ((pred==1)&(y_bin==0)).sum()
        nb = tp/n - fp/n * (pt/(1-pt))
        nb_all = prevalence - (1-prevalence)*(pt/(1-pt))
        rows.append({'threshold_probability':pt, 'net_benefit_model':nb, 'net_benefit_treat_all':nb_all, 'net_benefit_treat_none':0.0})
    return pd.DataFrame(rows)

dca_df = decision_curve(test_y, FINAL_PROBS)
dca_df.to_csv(TABLE_DIR/'table05_malignant_decision_curve.csv', index=False)


,coverage,retained_n,referred_n,accuracy,balanced_accuracy,macro_f1,malignant_sensitivity,ece
0,1.00,752,0,0.555851,0.607030,0.504733,0.794118,0.072101
1,0.95,714,38,0.568627,0.618379,0.517433,0.803030,0.076743
2,0.90,677,75,0.574594,0.633937,0.524523,0.836066,0.080210
3,0.85,639,113,0.586854,0.654992,0.538867,0.879310,0.081009
4,0.80,602,150,0.598007,0.659940,0.549162,0.877193,0.089447
5,0.75,564,188,0.602837,0.668532,0.553800,0.884615,0.092361
6,0.70,526,226,0.612167,0.675145,0.566541,0.884615,0.100691
7,0.60,451,301,0.627494,0.692293,0.586456,0.918367,0.117911


In [14]:
# ============================================================
# Cell 14 — Portion 5: publication visual helpers and core curves
# ============================================================
def plot_confusion(y, probs, name, title, normalize=False):
    pred = probs.argmax(axis=1)
    cm = confusion_matrix(y, pred, labels=[0,1,2])
    mat = cm.astype(float)
    if normalize:
        mat = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(mat, aspect='auto')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            txt = f'{mat[i,j]:.2f}' if normalize else str(int(cm[i,j]))
            ax.text(j,i,txt,ha='center',va='center',color='white' if mat[i,j] > mat.max()/2 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(name)

# fig06-09 core evaluation
plot_confusion(test_y, FINAL_PROBS, 'fig06_final_confusion_matrix.png', 'Final confusion matrix', normalize=False)
plot_confusion(test_y, FINAL_PROBS, 'fig06b_final_confusion_matrix_normalized.png', 'Final normalized confusion matrix', normalize=True)

y_bin = label_binarize(test_y, classes=[0,1,2])
fig, ax = plt.subplots(figsize=(6.8,5))
for i,c in enumerate(CLASS_NAMES):
    try:
        fpr,tpr,_ = roc_curve(y_bin[:,i], FINAL_PROBS[:,i])
        auc = roc_auc_score(y_bin[:,i], FINAL_PROBS[:,i])
        ax.plot(fpr,tpr,label=f'{c} AUC={auc:.3f}')
    except Exception: pass
ax.plot([0,1],[0,1],'--',linewidth=1)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('One-vs-rest ROC curves')
ax.legend(loc='lower right')
savefig('fig07_final_roc_curves.png')

fig, ax = plt.subplots(figsize=(6.8,5))
for i,c in enumerate(CLASS_NAMES):
    try:
        pr,rc,_ = precision_recall_curve(y_bin[:,i], FINAL_PROBS[:,i])
        ap = average_precision_score(y_bin[:,i], FINAL_PROBS[:,i])
        ax.plot(rc,pr,label=f'{c} AP={ap:.3f}')
    except Exception: pass
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('One-vs-rest precision-recall curves')
ax.legend(loc='lower left')
savefig('fig08_final_precision_recall_curves.png')

# Reliability diagram
fig, ax = plt.subplots(figsize=(6.5,5))
conf = FINAL_PROBS.max(axis=1); pred = FINAL_PROBS.argmax(axis=1); corr = (pred==test_y).astype(float)
bins=np.linspace(0,1,11); xs=[]; ys=[]; ns=[]
for i in range(len(bins)-1):
    mask=(conf>=bins[i])&(conf<bins[i+1] if i<len(bins)-2 else conf<=bins[i+1])
    if mask.any():
        xs.append(conf[mask].mean()); ys.append(corr[mask].mean()); ns.append(mask.sum())
ax.plot([0,1],[0,1],'--',label='perfect calibration')
ax.plot(xs,ys,marker='o',label=f'ECE={ece_score(test_y, FINAL_PROBS):.3f}')
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('Mean confidence'); ax.set_ylabel('Empirical accuracy')
ax.set_title('Reliability diagram')
ax.legend()
savefig('fig09_final_reliability_diagram.png')

# Confidence histogram
fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(conf[corr==1], bins=20, alpha=0.65, label='correct', edgecolor='black')
ax.hist(conf[corr==0], bins=20, alpha=0.65, label='error', edgecolor='black')
ax.set_xlabel('Prediction confidence'); ax.set_ylabel('Count')
ax.set_title('Confidence distribution for correct vs incorrect predictions')
ax.legend()
savefig('fig10_confidence_histogram.png')

# Selective referral
fig, ax = plt.subplots(figsize=(7,4.5))
ax.plot(selective_df['coverage'], selective_df['accuracy'], marker='o', label='accuracy')
ax.plot(selective_df['coverage'], selective_df['macro_f1'], marker='o', label='macro-F1')
ax.plot(selective_df['coverage'], selective_df['malignant_sensitivity'], marker='o', label='malignant sensitivity')
ax.invert_xaxis()
ax.set_xlabel('Coverage retained')
ax.set_ylabel('Metric')
ax.set_title('Selective referral / risk-coverage analysis')
ax.legend()
savefig('fig11_selective_referral_risk_coverage.png')


PosixPath('/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/figures/fig11_selective_referral_risk_coverage.png')

In [15]:
# ============================================================
# Cell 15 — Portion 5: metric panels, bootstrap, distributions, thresholds
# ============================================================
# fig12 metric panel
panel_metrics = ['accuracy','balanced_accuracy','macro_f1','malignant_sensitivity','malignant_specificity','malignant_auprc','macro_auroc','ece']
vals = [final_m.get(k,np.nan) for k in panel_metrics]
fig, ax = plt.subplots(figsize=(10,4.8))
ax.bar(range(len(panel_metrics)), vals, edgecolor='black')
ax.set_xticks(range(len(panel_metrics))); ax.set_xticklabels(panel_metrics, rotation=35, ha='right')
ax.set_ylabel('Value')
ax.set_title('Final metric panel')
for i,v in enumerate(vals):
    if np.isfinite(v): ax.text(i, v+0.015 if v<0.96 else v-0.05, f'{v:.3f}', ha='center')
savefig('fig12_final_metric_panel.png')

# fig13 per-class profile
per_rows=[]
for c in CLASS_NAMES:
    per_rows.append({'class':c,'sensitivity':final_m[f'{c}_sensitivity'],'specificity':final_m[f'{c}_specificity'],'precision':final_m[f'{c}_precision'],'f1':final_m[f'{c}_f1'],'auprc':final_m.get(f'{c}_auprc',np.nan)})
per_df=pd.DataFrame(per_rows); per_df.to_csv(TABLE_DIR/'table06_per_class_metrics.csv',index=False)
fig, ax = plt.subplots(figsize=(9,4.8))
x=np.arange(len(CLASS_NAMES)); width=0.16
for j,mn in enumerate(['sensitivity','specificity','precision','f1','auprc']):
    ax.bar(x+(j-2)*width, per_df[mn], width, label=mn, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0,1.05); ax.set_ylabel('Metric')
ax.set_title('Per-class metric profile')
ax.legend(ncol=3)
savefig('fig13_per_class_metric_profile.png')

# fig14 bootstrap CI forest
fig, ax = plt.subplots(figsize=(8,6))
plot_ci = ci_df.iloc[::-1].copy()
ypos=np.arange(len(plot_ci))
ax.errorbar(plot_ci['estimate'], ypos, xerr=[plot_ci['estimate']-plot_ci['ci_low'], plot_ci['ci_high']-plot_ci['estimate']], fmt='o', capsize=3)
ax.set_yticks(ypos); ax.set_yticklabels(plot_ci['metric'])
ax.set_xlabel('Metric value')
ax.set_title('Bootstrap 95% confidence intervals')
savefig('fig14_bootstrap_ci_forest.png')

# fig15 probability distributions by true class
fig, ax = plt.subplots(figsize=(8,4.8))
for i,c in enumerate(CLASS_NAMES):
    vals = FINAL_PROBS[test_y==i, i]
    if len(vals): ax.hist(vals, bins=20, alpha=0.45, label=f'true {c}', edgecolor='black')
ax.set_xlabel('Predicted probability assigned to true class')
ax.set_ylabel('Count')
ax.set_title('True-class probability distributions')
ax.legend()
savefig('fig15_probability_distributions_by_true_class.png')

# fig16 malignant probability histogram
fig, ax = plt.subplots(figsize=(7,4.5))
for i,c in enumerate(CLASS_NAMES):
    vals = FINAL_PROBS[test_y==i, 2]
    if len(vals): ax.hist(vals, bins=25, alpha=0.45, label=f'true {c}', edgecolor='black')
ax.set_xlabel('Predicted malignant probability')
ax.set_ylabel('Count')
ax.set_title('Malignant probability separation')
ax.legend()
savefig('fig16_malignant_probability_histogram.png')

# fig18 confidence correct vs error boxplot
fig, ax = plt.subplots(figsize=(5.6,4.5))
ax.boxplot([conf[corr==1], conf[corr==0]], labels=['correct','error'], showfliers=False)
ax.set_ylabel('Confidence')
ax.set_title('Confidence by prediction correctness')
savefig('fig18_confidence_correct_vs_error_boxplot.png')

# fig19 decision curve
fig, ax = plt.subplots(figsize=(7,4.8))
ax.plot(dca_df['threshold_probability'], dca_df['net_benefit_model'], label='model')
ax.plot(dca_df['threshold_probability'], dca_df['net_benefit_treat_all'], '--', label='treat all')
ax.plot(dca_df['threshold_probability'], dca_df['net_benefit_treat_none'], ':', label='treat none')
ax.set_xlabel('Threshold probability')
ax.set_ylabel('Net benefit')
ax.set_title('Decision curve analysis for malignant triage')
ax.legend()
savefig('fig19_malignant_decision_curve.png')

# fig29 threshold sweep
fig, ax = plt.subplots(figsize=(7,4.8))
for col in ['accuracy','macro_f1','malignant_sensitivity','malignant_specificity']:
    ax.plot(thr_df['threshold'], thr_df[col], label=col)
ax.set_xlabel('Malignant probability threshold')
ax.set_ylabel('Metric')
ax.set_title('Malignant threshold sweep')
ax.legend()
savefig('fig29_malignant_threshold_sweep.png')


PosixPath('/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/figures/fig29_malignant_threshold_sweep.png')

In [16]:
# ============================================================
# Cell 16 — Portion 5: high-confidence error gallery and XAI attention audit
# ============================================================
# Build test dataframe in prediction order.
test_indices = seed_outputs[0]['test_indices']
test_df_pred = splits['test'].reset_index(drop=True).iloc[test_indices].reset_index(drop=True) if len(test_indices)==len(test_y) else splits['test'].reset_index(drop=True)
test_df_pred['true_id'] = test_y
test_df_pred['true_label'] = [ID_TO_CLASS[int(i)] for i in test_y]
test_df_pred['pred_id'] = FINAL_PROBS.argmax(axis=1)
test_df_pred['pred_label'] = [ID_TO_CLASS[int(i)] for i in test_df_pred['pred_id']]
test_df_pred['confidence'] = FINAL_PROBS.max(axis=1)
for i,c in enumerate(CLASS_NAMES): test_df_pred[f'prob_{c}'] = FINAL_PROBS[:,i]
test_df_pred.to_csv(TABLE_DIR/'table07_test_predictions.csv', index=False)

# fig17 high-confidence error gallery
err = test_df_pred[test_df_pred['true_id'] != test_df_pred['pred_id']].sort_values('confidence', ascending=False).head(12)
fig, axes = plt.subplots(3,4, figsize=(12,8))
axes = axes.ravel()
for ax, (_,row) in zip(axes, err.iterrows()):
    try:
        im = Image.open(row['image_path']).convert('L')
        ax.imshow(im, cmap='gray')
        ax.set_title(f"T:{row['true_label']} | P:{row['pred_label']}\nconf={row['confidence']:.2f}", fontsize=9)
    except Exception:
        ax.text(0.5,0.5,'image error',ha='center')
    ax.axis('off')
for ax in axes[len(err):]: ax.axis('off')
fig.suptitle('High-confidence error gallery', y=0.98, fontsize=14)
savefig('fig17_high_confidence_error_gallery.png')

# XAI attention/mask audit from averaged first available TrustNet predictions.
att_arrays = [o.get('trust_test_attn') for o in seed_outputs if o.get('trust_test_attn') is not None]
mask_pred_arrays = [o.get('trust_test_mask_pred') for o in seed_outputs if o.get('trust_test_mask_pred') is not None]
ATT = np.mean(att_arrays, axis=0) if att_arrays else None
MASKP = np.mean(mask_pred_arrays, axis=0) if mask_pred_arrays else None

def load_mask_np(path, size):
    try:
        if isinstance(path, str) and Path(path).exists():
            m = Image.open(path).convert('L')
            m = TF.resize(m, [size,size], interpolation=TF.InterpolationMode.NEAREST)
            arr = np.array(m).astype(np.float32)/255.0
            return (arr > 0.5).astype(np.float32)
    except Exception:
        pass
    return np.zeros((size,size), dtype=np.float32)

xai_rows=[]
if ATT is not None:
    size = CFG['image_size']
    for i,row in test_df_pred.iterrows():
        if row['true_id'] == 0: continue
        gt = load_mask_np(row.get('mask_path', None), size)
        if gt.sum() <= 0: continue
        att = ATT[i,0]
        att = (att - att.min())/(att.max()-att.min()+1e-8)
        pred_mask = (MASKP[i,0] > 0.5).astype(np.float32) if MASKP is not None else (att > 0.5).astype(np.float32)
        energy = float((att*gt).sum()/(att.sum()+1e-8))
        peak_inside = float(gt[np.unravel_index(np.argmax(att), att.shape)] > 0)
        inter = (pred_mask*gt).sum(); union = ((pred_mask+gt)>0).sum()
        iou = float(inter/(union+1e-8))
        dice = float(2*inter/(pred_mask.sum()+gt.sum()+1e-8))
        xai_rows.append({'row_index':i,'true_label':row['true_label'],'pred_label':row['pred_label'],'energy_inside_mask':energy,'pointing_accuracy':peak_inside,'mask_iou':iou,'mask_dice':dice})

xai_df = pd.DataFrame(xai_rows)
xai_df.to_csv(TABLE_DIR/'table08_xai_attention_mask_audit.csv', index=False)
if len(xai_df):
    display(xai_df.describe())

# fig22 XAI overlay grid
sel = test_df_pred[(test_df_pred['true_id']>0) & test_df_pred.get('has_mask', pd.Series(False,index=test_df_pred.index)).astype(bool)].head(12)
fig, axes = plt.subplots(3,4, figsize=(12,8))
axes = axes.ravel()
for ax, (idx,row) in zip(axes, sel.iterrows()):
    try:
        im = Image.open(row['image_path']).convert('L')
        im = TF.resize(im, [CFG['image_size'], CFG['image_size']])
        arr = np.array(im)
        ax.imshow(arr, cmap='gray')
        if ATT is not None:
            att = ATT[idx,0]
            att = (att-att.min())/(att.max()-att.min()+1e-8)
            ax.imshow(att, alpha=0.35)
        gt = load_mask_np(row.get('mask_path', None), CFG['image_size'])
        ax.contour(gt, levels=[0.5], linewidths=1.2)
        ax.set_title(f"T:{row['true_label']} P:{row['pred_label']}", fontsize=9)
    except Exception:
        ax.text(0.5,0.5,'xai error',ha='center')
    ax.axis('off')
for ax in axes[len(sel):]: ax.axis('off')
fig.suptitle('XAI attention overlay with tumor-mask boundary', y=0.98, fontsize=14)
savefig('fig22_xai_attention_mask_overlay_grid.png')

# fig23 and fig24 XAI metrics
if len(xai_df):
    fig, ax = plt.subplots(figsize=(6,4.5))
    xai_df.boxplot(column='energy_inside_mask', by='true_label', ax=ax)
    plt.suptitle('')
    ax.set_title('Attention energy inside tumor mask')
    ax.set_xlabel('True tumor class'); ax.set_ylabel('Energy inside mask')
    savefig('fig23_xai_energy_inside_mask_boxplot.png')

    means = xai_df[['energy_inside_mask','pointing_accuracy','mask_iou','mask_dice']].mean()
    fig, ax = plt.subplots(figsize=(7,4.5))
    ax.bar(means.index, means.values, edgecolor='black')
    ax.set_ylim(0,1.05)
    ax.set_title('XAI and segmentation audit metrics')
    ax.set_ylabel('Mean value')
    for i,v in enumerate(means.values): ax.text(i, v+0.03, f'{v:.3f}', ha='center')
    plt.xticks(rotation=25, ha='right')
    savefig('fig24_xai_segmentation_metric_panel.png')
else:
    # create placeholder figure for manifest completeness
    fig, ax = plt.subplots(figsize=(6,4))
    ax.axis('off'); ax.text(0.5,0.5,'No valid tumor masks available for XAI audit',ha='center',va='center')
    savefig('fig23_xai_energy_inside_mask_boxplot.png')
    fig, ax = plt.subplots(figsize=(6,4))
    ax.axis('off'); ax.text(0.5,0.5,'No XAI metrics available',ha='center',va='center')
    savefig('fig24_xai_segmentation_metric_panel.png')


,row_index,energy_inside_mask,pointing_accuracy,mask_iou,mask_dice
count,372.000000,372.000000,372.000000,372.000000,372.000000
mean,371.387097,0.032033,0.045699,0.021267,0.033283
std,215.504425,0.050431,0.209113,0.076309,0.112844
min,3.000000,0.000024,0.000000,0.000000,0.000000
25%,183.250000,0.001840,0.000000,0.000000,0.000000
50%,364.500000,0.009602,0.000000,0.000000,0.000000
75%,559.250000,0.038572,0.000000,0.000000,0.000000
max,746.000000,0.384687,1.000000,0.559520,0.717554


In [17]:
# ============================================================
# Cell 17 — Portion 5: additional journal-ready visuals up to 36+
# ============================================================
# fig20 baseline vs final comparison if P2 baseline metrics exist.
def find_baseline_metrics():
    candidates=[]
    for p in table_paths:
        name=p.name.lower()
        if 'baseline' in name and 'metric' in name and p.suffix.lower()=='.csv':
            candidates.append(p)
    for p in candidates:
        df=read_table(p)
        if df is not None and len(df):
            return p, df
    return None, None

bp, bdf = find_baseline_metrics()
fig, ax = plt.subplots(figsize=(8,4.8))
compare_metrics = ['accuracy','balanced_accuracy','macro_f1','malignant_sensitivity','malignant_auprc','ece']
final_vals=[final_m.get(k,np.nan) for k in compare_metrics]
if bdf is not None:
    # find best baseline row by macro_f1 if possible
    b = bdf.copy()
    lower = {c.lower():c for c in b.columns}
    vals=[]
    for k in compare_metrics:
        c = lower.get(k) or lower.get(k.replace('_',' '))
        if c is not None:
            vals.append(pd.to_numeric(b[c], errors='coerce').max())
        else:
            vals.append(np.nan)
else:
    vals=[np.nan]*len(compare_metrics)
x=np.arange(len(compare_metrics)); width=0.35
ax.bar(x-width/2, vals, width, label='best available baseline', edgecolor='black')
ax.bar(x+width/2, final_vals, width, label='final model', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(compare_metrics, rotation=30, ha='right')
ax.set_title('Baseline vs final model comparison')
ax.set_ylabel('Metric')
ax.legend()
savefig('fig20_baseline_vs_final_comparison.png')

# fig21 subgroup robustness by safe metadata if available
sub_rows=[]
for c in META_CAT_COLS[:3]:
    if c in test_df_pred.columns:
        for val, grp in test_df_pred.groupby(c):
            if len(grp) >= 15:
                idx = grp.index.values
                mm = all_metrics(test_y[idx], FINAL_PROBS[idx])
                sub_rows.append({'subgroup_col':c,'subgroup':str(val),'n':len(grp),'accuracy':mm['accuracy'],'macro_f1':mm['macro_f1'],'malignant_sensitivity':mm['malignant_sensitivity']})
sub_df=pd.DataFrame(sub_rows)
sub_df.to_csv(TABLE_DIR/'table09_subgroup_robustness.csv',index=False)
fig, ax = plt.subplots(figsize=(8,4.8))
if len(sub_df):
    lab = (sub_df['subgroup_col']+':'+sub_df['subgroup']).tolist()
    ax.bar(range(len(sub_df)), sub_df['macro_f1'], edgecolor='black')
    ax.set_xticks(range(len(sub_df))); ax.set_xticklabels(lab, rotation=45, ha='right')
    ax.set_ylabel('Macro-F1')
    ax.set_title('Subgroup robustness')
else:
    ax.axis('off'); ax.text(0.5,0.5,'No safe metadata subgroup available\n(leakage guard active)',ha='center',va='center')
savefig('fig21_subgroup_robustness.png')

# fig25 feature space t-SNE/PCA
features = np.mean([o.get('trust_test_features') for o in seed_outputs if o.get('trust_test_features') is not None], axis=0)
fig, ax = plt.subplots(figsize=(6.5,5.5))
try:
    X = features
    if len(X) > 900:
        rng=np.random.default_rng(42); idx=rng.choice(len(X),900,replace=False)
    else:
        idx=np.arange(len(X))
    Xs=X[idx]; ys=test_y[idx]
    if Xs.shape[1] > 50:
        Xs = PCA(n_components=50, random_state=42).fit_transform(Xs)
    emb = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto', perplexity=min(30, max(5,len(idx)//20))).fit_transform(Xs)
    for i,c in enumerate(CLASS_NAMES):
        m=ys==i; ax.scatter(emb[m,0], emb[m,1], s=18, label=c, alpha=0.75)
    ax.set_title('Feature-space visualization')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend()
except Exception as e:
    ax.axis('off'); ax.text(0.5,0.5,'Feature visualization unavailable\n'+str(e)[:100],ha='center')
savefig('fig25_tsne_feature_space.png')

# fig26 classwise calibration curves
fig, ax = plt.subplots(figsize=(7,5))
for i,c in enumerate(CLASS_NAMES):
    y_c=(test_y==i).astype(float); p_c=FINAL_PROBS[:,i]
    xs=[]; ys=[]
    for lo,hi in zip(np.linspace(0,1,11)[:-1], np.linspace(0,1,11)[1:]):
        m=(p_c>=lo)&(p_c<hi if hi<1 else p_c<=hi)
        if m.any(): xs.append(p_c[m].mean()); ys.append(y_c[m].mean())
    ax.plot(xs,ys,marker='o',label=c)
ax.plot([0,1],[0,1],'--',linewidth=1)
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed frequency')
ax.set_title('Classwise calibration curves')
ax.legend()
savefig('fig26_classwise_calibration_curves.png')

# fig27 class score histograms
fig, axes = plt.subplots(1,3, figsize=(14,4))
for i,c in enumerate(CLASS_NAMES):
    axes[i].hist(FINAL_PROBS[:,i], bins=25, edgecolor='black')
    axes[i].set_title(f'{c} probability')
    axes[i].set_xlabel('Probability'); axes[i].set_ylabel('Count')
savefig('fig27_class_score_histograms.png')

# fig28 entropy by true class
entropy = -np.sum(FINAL_PROBS*np.log(np.clip(FINAL_PROBS,1e-12,1)), axis=1)
fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.boxplot([entropy[test_y==i] for i in range(3)], labels=CLASS_NAMES, showfliers=False)
ax.set_ylabel('Predictive entropy')
ax.set_title('Uncertainty by true class')
savefig('fig28_entropy_by_true_class.png')

# fig30 cumulative gains + fig31 lift curve for malignant
order=np.argsort(-FINAL_PROBS[:,2]); ymal=(test_y==2).astype(int)[order]
cum=np.cumsum(ymal); total=max(ymal.sum(),1); frac=np.arange(1,len(ymal)+1)/len(ymal)
gain=cum/total
fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.plot(frac,gain,label='model'); ax.plot([0,1],[0,1],'--',label='random')
ax.set_xlabel('Fraction of reviewed cases'); ax.set_ylabel('Fraction of malignant cases captured')
ax.set_title('Malignant cumulative gains curve'); ax.legend()
savefig('fig30_malignant_cumulative_gains.png')
fig, ax = plt.subplots(figsize=(6.5,4.5))
lift=np.divide(gain, frac, out=np.zeros_like(gain), where=frac>0)
ax.plot(frac,lift); ax.axhline(1, linestyle='--', linewidth=1)
ax.set_xlabel('Fraction of reviewed cases'); ax.set_ylabel('Lift over random')
ax.set_title('Malignant lift curve')
savefig('fig31_malignant_lift_curve.png')

# fig32 normal vs tumor ROC
fig, ax = plt.subplots(figsize=(6.2,4.8))
y_tum=(test_y>0).astype(int); p_tum=FINAL_PROBS[:,1]+FINAL_PROBS[:,2]
try:
    fpr,tpr,_=roc_curve(y_tum,p_tum); auc=roc_auc_score(y_tum,p_tum)
    ax.plot(fpr,tpr,label=f'AUC={auc:.3f}'); ax.plot([0,1],[0,1],'--')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
    ax.set_title('Normal-vs-tumor screening ROC'); ax.legend()
except Exception as e:
    ax.axis('off'); ax.text(0.5,0.5,str(e),ha='center')
savefig('fig32_normal_vs_tumor_roc.png')

# fig33/34 tumor-only benign-vs-malignant curves
mask_tum=test_y>0
fig, ax = plt.subplots(figsize=(6.2,4.8))
try:
    ybm=(test_y[mask_tum]==2).astype(int); pbm=FINAL_PROBS[mask_tum,2]/np.maximum(FINAL_PROBS[mask_tum,1]+FINAL_PROBS[mask_tum,2],1e-8)
    fpr,tpr,_=roc_curve(ybm,pbm); auc=roc_auc_score(ybm,pbm)
    ax.plot(fpr,tpr,label=f'AUC={auc:.3f}'); ax.plot([0,1],[0,1],'--')
    ax.set_title('Tumor-only benign-vs-malignant ROC'); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend()
except Exception as e:
    ax.axis('off'); ax.text(0.5,0.5,str(e),ha='center')
savefig('fig33_tumor_only_benign_malignant_roc.png')
fig, ax = plt.subplots(figsize=(6.2,4.8))
try:
    pr,rc,_=precision_recall_curve(ybm,pbm); ap=average_precision_score(ybm,pbm)
    ax.plot(rc,pr,label=f'AP={ap:.3f}')
    ax.set_title('Tumor-only malignant precision-recall'); ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend()
except Exception as e:
    ax.axis('off'); ax.text(0.5,0.5,str(e),ha='center')
savefig('fig34_tumor_only_malignant_pr.png')

# fig35 classwise false negative profile
fn_vals=[final_m[f'{c}_fnr'] for c in CLASS_NAMES]
fig, ax = plt.subplots(figsize=(6.2,4.5))
ax.bar(CLASS_NAMES, fn_vals, edgecolor='black')
ax.set_ylabel('False negative rate')
ax.set_title('Classwise false-negative profile')
for i,v in enumerate(fn_vals): ax.text(i, v+0.02, f'{v:.3f}', ha='center')
savefig('fig35_classwise_false_negative_profile.png')

# fig36 confidence quantile accuracy
qdf=pd.DataFrame({'conf':conf,'correct':corr})
qdf['quantile']=pd.qcut(qdf['conf'], q=min(10, max(2, qdf['conf'].nunique())), duplicates='drop')
qsum=qdf.groupby('quantile').agg(mean_conf=('conf','mean'),accuracy=('correct','mean'),n=('correct','size')).reset_index()
qsum.to_csv(TABLE_DIR/'table10_confidence_quantile_accuracy.csv',index=False)
fig, ax = plt.subplots(figsize=(7,4.5))
ax.plot(range(len(qsum)), qsum['accuracy'], marker='o', label='accuracy')
ax.plot(range(len(qsum)), qsum['mean_conf'], marker='o', label='mean confidence')
ax.set_xticks(range(len(qsum))); ax.set_xticklabels([str(x) for x in qsum['quantile']], rotation=45, ha='right')
ax.set_title('Confidence quantile accuracy')
ax.set_ylabel('Value'); ax.legend()
savefig('fig36_confidence_quantile_accuracy.png')

print('Saved extended journal visuals to:', FIG_DIR)


Saved extended journal visuals to: /kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5/figures


In [18]:
# ============================================================
# Cell 18 — Final report, figure manifest, and ZIP export
# ============================================================
fig_manifest = []
for p in sorted(FIG_DIR.glob('*.png')):
    fig_manifest.append({'figure':p.name, 'path':str(p), 'size_kb':round(p.stat().st_size/1024,1)})
fig_manifest_df = pd.DataFrame(fig_manifest)
fig_manifest_df.to_csv(TABLE_DIR/'table11_figure_manifest.csv', index=False)

metric_lines = []
for k in ['accuracy','balanced_accuracy','macro_f1','malignant_sensitivity','malignant_specificity','malignant_auprc','macro_auroc','ece','nll','mcc','cohen_kappa']:
    metric_lines.append(f'- **{k}**: {final_m.get(k, np.nan):.4f}')

report = f'''
# BTXRD Portion 3/4/5 Final T4-Efficient SOTA Report

## Dataset and split audit

Final usable dataset count:

{final_counts.to_frame('count').to_markdown()}

Internal split counts:

{final_master.groupby(['split','label_3class']).size().unstack(fill_value=0).reindex(columns=CLASS_NAMES, fill_value=0).to_markdown()}

## Model

**BTX-TrustNet-LiteSOTA-v5** combines an EfficientNet-B3 global expert with a lesion-aware TrustNet expert. The final prediction uses validation-only ensemble weighting, temperature calibration, and logit-offset operating point tuning.

## Best final test metrics

{chr(10).join(metric_lines)}

## Calibration settings

- TrustNet ensemble weight: **{w:.3f}**
- Temperature: **{T:.3f}**
- Logit offset: **{offset.tolist()}**

## Selective referral summary

{selective_df.to_markdown(index=False)}

## Main output files

- `tables/table01_final_30plus_metrics.csv`
- `tables/table02_bootstrap_95ci.csv`
- `tables/table03_selective_referral.csv`
- `tables/table07_test_predictions.csv`
- `tables/table08_xai_attention_mask_audit.csv`
- `figures/`
- `models/`
- `arrays/`

## Reporting note

Use `primary_calibrated` as the strict blind-test calibrated result. Use the logit-offset result as a validation-tuned clinical operating point. Do not claim prospective clinical validation from this retrospective dataset.
'''
(REPORT_DIR/'BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5_Report.md').write_text(report)
print(report[:1800])

# Save final summary table.
summary = pd.DataFrame({
    'item':['output_dir','report','zip','num_figures','run_mode','image_size','batch_size','seeds'],
    'value':[str(OUT_DIR), str(REPORT_DIR/'BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5_Report.md'), str(OUT_DIR)+'.zip', len(fig_manifest_df), RUN_MODE, CFG['image_size'], CFG['batch_size'], str(CFG['seeds'])]
})
summary.to_csv(TABLE_DIR/'table12_final_output_summary.csv', index=False)
display(summary)

# Zip outputs.
zip_path = shutil.make_archive(str(OUT_DIR), 'zip', root_dir=OUT_DIR)
print('Final ZIP:', zip_path)



# BTXRD Portion 3/4/5 Final T4-Efficient SOTA Report

## Dataset and split audit

Final usable dataset count:

| label_3class   |   count |
|:---------------|--------:|
| normal         |    1899 |
| benign         |    1517 |
| malignant      |     341 |

Internal split counts:

| split   |   normal |   benign |   malignant |
|:--------|---------:|---------:|------------:|
| test    |      380 |      304 |          68 |
| train   |     1329 |     1061 |         239 |
| val     |      190 |      152 |          34 |

## Model

**BTX-TrustNet-LiteSOTA-v5** combines an EfficientNet-B3 global expert with a lesion-aware TrustNet expert. The final prediction uses validation-only ensemble weighting, temperature calibration, and logit-offset operating point tuning.

## Best final test metrics

- **accuracy**: 0.5559
- **balanced_accuracy**: 0.6070
- **macro_f1**: 0.5047
- **malignant_sensitivity**: 0.7941
- **malignant_specificity**: 0.7485
- **malignant_auprc**: 0.4628
- **macro_auroc**: 0.7

,item,value
0,output_dir,/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINA...
1,report,/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINA...
2,zip,/kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINA...
3,num_figures,36
4,run_mode,publication
5,image_size,320
6,batch_size,16
7,seeds,"[42, 909]"


Final ZIP: /kaggle/working/BTXRD_P3_4_5_T4_EFFICIENT_FINAL_V5.zip
